# AGOS | Manila Barangay Flood-Risk Prioritization

**Study:** Barangay-Level Flood Risk Classification and Mapping in the City of Manila Using Machine Learning and a Multi-Criteria Disaster Prioritization Index  

**Scope:** 897 Manila barangays · 5-year and 25-year LiPAD hazard layers · PSA 2020/2024 population · NAMRIA-derived elevation  

**Output:** Research-based **Low / Medium / High** planning priorities; **not** official hazard ratings, real-time flood warnings, or forecasts.


## Files Needed

Because the shape-to-CSV/geospatial processing was completed separately in QGIS, this notebook now requires only tabular inputs:

1. `MetroManila_Barangay_Flood_Area_Validated.csv` — QGIS-prepared flood area and flood percentage table.
2. `population-dataset-2024.csv` — PSA 2024 barangay population estimates.
3. `population-dataset-2020.xlsx` — PSA 2020 barangay population table.
4. `manila_final_base_dataset_NEW.csv` — optional fallback for 2020 population if Excel parsing yields no rows; this notebook does not otherwise join its administrative fields.
5. `manila_barangay_elevation.csv` — required QGIS Zonal Statistics output containing barangay-level DEM/DTM elevation statistics.

### A Note on Elevation

Elevation is treated as a **vulnerability input**. The elevation table must contain barangay-level mean elevation for all 897 Manila barangays. The expected QGIS fields are:

- `adm4_name`
- `Elev_count`
- `Elev_mean`
- `Elev_median`
- `Elev_min`
- `Elev_max`

These are standardized in the notebook as:

- `Barangay`
- `Elevation_Count`
- `Elevation_Mean`
- `Elevation_Median`
- `Elevation_Min`
- `Elevation_Max`

`Elevation_Mean` is used in the Vulnerability Score and is reverse-normalized because lower elevation corresponds to higher flood susceptibility.


## Outputs

| File in `outputs/` | Purpose |
|---|---|
| `MetroManila_Barangay_Flood_Area_Validated.csv` | Copy of checked QGIS flood table |
| `manila-city-flood-population_enriched_NEW.csv` / `.xlsx` | Merged analytical dataset (Excel depends on `openpyxl`) |
| `manila_ml_ready_dataset.csv` | Exactly 11 predictors and index-derived class target |
| `model_comparison_results.csv` | RF, GB, and MLP comparison |
| `barangay_flood_risk_predictions.csv` | Single 897-barangay results CSV for use by the AGOS prototype |
| `best_flood_risk_model_pipeline.joblib` | Server-side sklearn model bundle; **do not serve to browsers** |

**Historical names (not the current export contract):** `manila_barangay_flood_spatial.csv`, `manila_dpi_rankings.csv`, `manila_model_comparison.csv`, `manila_feature_importance.csv`, `best_model_pipeline.joblib`. `manila_barangay_elevation.csv` is an **input**, not a produced output. Retained here to help reconcile older consultation documents; do not claim these files are generated in the current run.


## 2. Library Imports and Configuration

**Purpose:** Import required libraries and set global analysis constants.

**Input:** None (environment packages).

**Output:** Configured Python session (display options, random state, expected barangay count, CRS, tolerances).

**Why Needed:** Reproducible thesis runs require a fixed random state, consistent numeric handling, and explicit study parameters (897 barangays; EPSG:32651 for area).


In [1]:
from __future__ import annotations

import os
import re
import warnings
from pathlib import Path
from typing import Iterable, Optional

import joblib
import numpy as np
import pandas as pd
import geopandas as gpd

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

EXPECTED_BARANGAY_COUNT = 897
AREA_EPSG = 32651  # UTM Zone 51N for Manila area calculations
CITY_FILTER = "CITY OF MANILA"
RANDOM_STATE = 42
EPS = 1e-6

EXPECTED_PSA_2020_TOTAL = 1_846_513
EXPECTED_PSA_2024_TOTAL = 1_902_590
EXPECTED_POP_CHANGE_ABS = 56_077
EXPECTED_POP_CHANGE_PCT = 3.04

KEEP_ONLY_NUMBERED_BARANGAYS = True
EXPECTED_MIN_BARANGAY_NO = 1
EXPECTED_MAX_BARANGAY_NO = 905
APPLY_HAZARD_ATTRIBUTE_FILTER = False
HAZARD_FILTER_RULES = {}

print("Libraries imported.")
print(f"Expected barangays: {EXPECTED_BARANGAY_COUNT}")
print(f"Area CRS: EPSG:{AREA_EPSG}")
print(f"Random state: {RANDOM_STATE}")

# Elevation extraction from NAMRIA DTM
try:
    import rasterio
    from rasterstats import zonal_stats
except ImportError:
    rasterio = None
    zonal_stats = None
    warnings.warn("Optional packages rasterio/rasterstats are unavailable. The notebook can still use a precomputed elevation CSV.")


Libraries imported.
Expected barangays: 897
Area CRS: EPSG:32651
Random state: 42


## 3. Input Path Configuration

**Purpose:** Define all required input and output paths in one place.

**Input:** User-editable paths to the QGIS-prepared flood table, population files, administrative base data, and required QGIS elevation table.

**Output:** Resolved `Path` objects used by every later stage.

**Why needed:** After paths are set correctly, the notebook should run top-to-bottom without further interaction.

This notebook assumes that spatial preprocessing was already completed in QGIS. Flood-area computation and elevation extraction are treated as upstream geospatial steps.

For area-based GIS calculations, EPSG:32651 was used where appropriate. The final CSV files are non-geometric tabular outputs.

In [2]:
# USER CONFIGURATION — TABULAR PIPELINE
# This notebook starts from CSV/XLSX tables exported from QGIS and other sources. It does not perform flood polygon intersections in this tabular workflow.

BASE_DIR = Path(".")
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Required Tabular Inputs
FLOOD_CSV = BASE_DIR / "MetroManila_Barangay_Flood_Area_Validated.csv"
PSA_2024_CSV = BASE_DIR / "population-dataset-2024.csv"
PSA_2020_XLSX = BASE_DIR / "population-dataset-2020.xlsx"
BASE_ADMIN_CSV = BASE_DIR / "manila_final_base_dataset_NEW.csv"

# Required QGIS Zonal Statistics elevation output
# Expected source file from QGIS: adm4_name, Elev_count, Elev_mean, Elev_median, Elev_min, Elev_max
ELEVATION_CSV = BASE_DIR / "manila_barangay_elevation.csv"
REQUIRE_ELEVATION = True

# Optional References
DTM_PATH = BASE_DIR / "DTM_ManilaCity_UTM51_PRS92.tif"
COMPUTE_ELEVATION_FROM_DTM_IF_AVAILABLE = False

# Outputs
OUT_FLOOD_CSV = OUTPUT_DIR / "MetroManila_Barangay_Flood_Area_Validated.csv"
OUT_ENRICHED_CSV = OUTPUT_DIR / "manila-city-flood-population_enriched_NEW.csv"
OUT_ENRICHED_XLSX = OUTPUT_DIR / "manila-city-flood-population_enriched_NEW.xlsx"
OUT_ML_READY_CSV = OUTPUT_DIR / "manila_ml_ready_dataset.csv"
OUT_MODEL_COMPARISON_CSV = OUTPUT_DIR / "model_comparison_results.csv"
OUT_PREDICTIONS_CSV = OUTPUT_DIR / "barangay_flood_risk_predictions.csv"
OUT_BEST_MODEL_JOBLIB = OUTPUT_DIR / "best_flood_risk_model_pipeline.joblib"

print("BASE_DIR:", BASE_DIR.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

for label, path in {
    "FLOOD_CSV": FLOOD_CSV,
    "PSA_2024_CSV": PSA_2024_CSV,
    "PSA_2020_XLSX": PSA_2020_XLSX,
    "BASE_ADMIN_CSV": BASE_ADMIN_CSV,
    "ELEVATION_CSV required": ELEVATION_CSV,
    "DTM_PATH reference only": DTM_PATH,
}.items():
    print(f"{label} exists:", path.exists(), "-", path)

if REQUIRE_ELEVATION and not ELEVATION_CSV.exists():
    raise FileNotFoundError(
        f"Required elevation file not found: {ELEVATION_CSV}. "
        "Use the QGIS Zonal Statistics output file or update ELEVATION_CSV."
    )

BASE_DIR: /content
OUTPUT_DIR: /content/outputs
FLOOD_CSV exists: True - MetroManila_Barangay_Flood_Area_Validated.csv
PSA_2024_CSV exists: True - population-dataset-2024.csv
PSA_2020_XLSX exists: True - population-dataset-2020.xlsx
BASE_ADMIN_CSV exists: True - manila_final_base_dataset_NEW.csv
ELEVATION_CSV required exists: True - manila_barangay_elevation.csv
DTM_PATH reference only exists: False - DTM_ManilaCity_UTM51_PRS92.tif


### Shared Helper Functions

**Purpose:** Centralize reusable operations used across spatial, merge, index, and ML stages.

**Input / Output:** Pure utility functions (name cleaning, normalization, entropy weights, validation summaries).

**Why Needed:** Thesis reproducibility and readability; repeated formulas (min-max, reverse min-max, entropy) must be applied consistently.


In [3]:
def clean_barangay_name(x) -> str:
    """Standardize barangay names for merges (e.g., Brgy. 105 -> BARANGAY 105)."""
    x = str(x).upper().strip()
    x = re.sub(r"\s+", " ", x)
    x = x.replace("BRGY.", "BARANGAY").replace("BRGY", "BARANGAY")
    return x


def clean_barangay_name_strict(x) -> str:
    """Strict merge key: remove spaces and hyphens (BARANGAY 202-A -> BARANGAY202A)."""
    return clean_barangay_name(x).replace("-", "").replace(" ", "")


def extract_barangay_number(name) -> Optional[int]:
    if pd.isna(name):
        return None
    match = re.search(r"Barangay\s+(\d+)", str(name), flags=re.IGNORECASE)
    return int(match.group(1)) if match else None


def extract_number(text) -> int:
    match = re.search(r"(\d+)", str(text))
    return int(match.group()) if match else 0


def is_numbered_barangay(name) -> bool:
    if pd.isna(name):
        return False
    return bool(re.fullmatch(r"Barangay\s+\d+(-[A-Z])?", str(name).strip(), flags=re.IGNORECASE))


def find_first_existing_column(df: pd.DataFrame, candidates: list, label: str) -> str:
    col = next((c for c in candidates if c in df.columns), None)
    if not col:
        raise ValueError(f"Could not find {label} column among {list(df.columns)}")
    return col


def keep_existing(cols: Iterable[str], df: pd.DataFrame) -> list:
    return [c for c in cols if c in df.columns]


def safe_numeric(df: pd.DataFrame, cols: Iterable[str]) -> pd.DataFrame:
    for col in cols:
        if col not in df.columns:
            continue
        if df[col].dtype == object:
            df[col] = (
                df[col].astype(str)
                .str.replace(",", "", regex=False)
                .str.replace("%", "", regex=False)
            )
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def minmax_normalize(series: pd.Series) -> pd.Series:
    """Min-max normalization: higher raw value -> higher normalized risk contribution."""
    series = pd.to_numeric(series, errors="coerce")
    min_val, max_val = series.min(), series.max()
    if pd.isna(min_val) or pd.isna(max_val) or max_val == min_val:
        return pd.Series(0.0, index=series.index)
    return (series - min_val) / (max_val - min_val)


def minmax_reverse(series: pd.Series) -> pd.Series:
    """Reverse min-max: lower raw value -> higher risk (used for elevation)."""
    series = pd.to_numeric(series, errors="coerce")
    min_val, max_val = series.min(), series.max()
    if pd.isna(min_val) or pd.isna(max_val) or max_val == min_val:
        return pd.Series(0.0, index=series.index)
    return (max_val - series) / (max_val - min_val)

def standardize_elevation_columns(elevation_df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize QGIS Zonal Statistics elevation column names.

    Accepts common QGIS/export variants and converts them to:
    Barangay, Elevation_Count, Elevation_Mean, Elevation_Median,
    Elevation_Min, Elevation_Max.
    """
    elevation_df = elevation_df.copy()
    elevation_df.columns = [str(c).strip() for c in elevation_df.columns]

    rename_map = {
        "adm4_name": "Barangay",
        "ADM4_NAME": "Barangay",
        "Barangay Name": "Barangay",
        "barangay": "Barangay",
        "Barangay": "Barangay",

        "Elev_count": "Elevation_Count",
        "ELEV_COUNT": "Elevation_Count",
        "Count": "Elevation_Count",
        "count": "Elevation_Count",

        "Elev_mean": "Elevation_Mean",
        "ELEV_MEAN": "Elevation_Mean",
        "Mean": "Elevation_Mean",
        "mean": "Elevation_Mean",
        "mean_elevation": "Elevation_Mean",
        "Elevation": "Elevation_Mean",

        "Elev_median": "Elevation_Median",
        "ELEV_MEDIAN": "Elevation_Median",
        "Median": "Elevation_Median",
        "median": "Elevation_Median",

        "Elev_min": "Elevation_Min",
        "ELEV_MIN": "Elevation_Min",
        "Min": "Elevation_Min",
        "min": "Elevation_Min",

        "Elev_max": "Elevation_Max",
        "ELEV_MAX": "Elevation_Max",
        "Max": "Elevation_Max",
        "max": "Elevation_Max",
    }

    elevation_df = elevation_df.rename(columns={
        col: rename_map[col]
        for col in elevation_df.columns
        if col in rename_map
    })

    return elevation_df


def validate_required_columns(df: pd.DataFrame, required_cols: list, dataset_name: str) -> None:
    """Raise a clear error if required columns are missing."""
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(
            f"{dataset_name} is missing required columns: {missing}. "
            f"Available columns: {list(df.columns)}"
        )

def entropy_weights(dataframe: pd.DataFrame):
    """Entropy weighting across component score columns; zeros handled safely for log."""
    x = dataframe.copy().replace([np.inf, -np.inf], np.nan).fillna(0).clip(lower=0)
    for col in x.columns:
        min_val, max_val = x[col].min(), x[col].max()
        x[col] = (x[col] - min_val) / (max_val - min_val) if max_val != min_val else 0.0

    col_sums = x.sum(axis=0)
    p = x.div(col_sums.replace(0, np.nan), axis=1).fillna(0)
    m = len(x)
    k = 1 / np.log(m) if m > 1 else 1.0
    p_log = p.replace(0, np.nan)
    entropy = (-k * (p * np.log(p_log)).sum(axis=0)).fillna(0)
    diversification = 1 - entropy
    if diversification.sum() == 0:
        weights = pd.Series(1 / len(diversification), index=diversification.index)
    else:
        weights = diversification / diversification.sum()
    return weights, entropy, diversification


def assert_exact_barangay_count(df: pd.DataFrame, stage: str, expected: int = EXPECTED_BARANGAY_COUNT) -> None:
    n = len(df)
    if n != expected:
        raise ValueError(
            f"[{stage}] Expected exactly {expected} barangays, found {n}. "
            "Check merges, filters, and input coverage for City of Manila."
        )
    print(f"[{stage}] Barangay count validated: {n}")


def validation_summary(title: str, checks: dict) -> pd.DataFrame:
    rows = [{"check": k, "count": int(v)} for k, v in checks.items()]
    report = pd.DataFrame(rows)
    print(f"\n=== {title} ===")
    print(report.to_string(index=False))
    return report


print("Helper functions defined.")



Helper functions defined.


### Required QGIS Elevation Data

Elevation is handled as a required tabular input in this notebook.

The barangay-level elevation statistics were generated upstream in QGIS using **Zonal Statistics**. The Manila barangay polygon layer was used as the zone layer, while the DEM/DTM raster was used as the elevation raster.

The expected QGIS output fields are:

- `adm4_name`
- `Elev_count`
- `Elev_mean`
- `Elev_median`
- `Elev_min`
- `Elev_max`

The notebook standardizes these fields as:

- `Barangay`
- `Elevation_Count`
- `Elevation_Mean`
- `Elevation_Median`
- `Elevation_Min`
- `Elevation_Max`

For the Disaster Prioritization Index, `Elevation_Mean` is used as the physical vulnerability indicator. It is reverse-normalized because lower elevation indicates greater flood susceptibility.

The optional DTM extraction helper below is retained only for reproducibility/reference. The final pipeline uses the QGIS-exported elevation CSV.


In [4]:
def extract_barangay_elevation_from_dtm(boundary_path, dtm_path, output_csv):
    """Extract mean/min/max/median elevation per barangay from DTM using zonal statistics."""
    if rasterio is None or zonal_stats is None:
        raise ImportError("Install rasterio and rasterstats to compute elevation from DTM inside the notebook.")

    boundary_path = Path(boundary_path)
    dtm_path = Path(dtm_path)
    output_csv = Path(output_csv)

    if not boundary_path.exists():
        raise FileNotFoundError(f"Barangay boundary file not found: {boundary_path}")
    if not dtm_path.exists():
        raise FileNotFoundError(f"DTM raster not found: {dtm_path}")

    gdf = gpd.read_file(boundary_path)
    gdf.columns = [str(c).strip() for c in gdf.columns]

    # Detect barangay name column
    name_col = next((c for c in ["Barangay", "barangay", "ADM4_EN", "adm4_en", "brgy", "brgy_name", "NAME_4"] if c in gdf.columns), None)
    if name_col is None:
        raise ValueError(f"Cannot detect barangay name column in boundary file. Columns: {list(gdf.columns)}")

    gdf = gdf.copy()
    gdf["Barangay"] = gdf[name_col].astype(str).str.strip()
    if KEEP_ONLY_NUMBERED_BARANGAYS:
        gdf = gdf[gdf["Barangay"].apply(is_numbered_barangay)].copy()

    if gdf.crs is None:
        raise ValueError("Barangay boundary file has no CRS. Define its CRS in QGIS before using it here.")

    # Repair geometries without doing any flood-intersection processing
    gdf["geometry"] = gdf.geometry.buffer(0)
    gdf = gdf[~gdf.geometry.is_empty & gdf.geometry.notna()].copy()

    with rasterio.open(dtm_path) as src:
        raster_crs = src.crs
        nodata = src.nodata

    if raster_crs is None:
        raise ValueError("DTM raster has no CRS. Define/provide a CRS in QGIS first.")

    if gdf.crs != raster_crs:
        gdf = gdf.to_crs(raster_crs)

    stats = zonal_stats(
        gdf,
        str(dtm_path),
        stats=["mean", "min", "max", "median"],
        nodata=nodata,
        all_touched=True
    )

    elev = pd.DataFrame(stats).rename(columns={
        "mean": "Elevation_Mean",
        "min": "Elevation_Min",
        "max": "Elevation_Max",
        "median": "Elevation_Median",
    })
    elev.insert(0, "Barangay", gdf["Barangay"].values)
    elev["barangay_key_strict"] = elev["Barangay"].apply(clean_barangay_name_strict)
    elev.to_csv(output_csv, index=False)
    print(f"Elevation CSV generated from DTM: {output_csv}")
    print("Rows:", len(elev), "Missing mean elevation:", int(elev["Elevation_Mean"].isna().sum()))
    return elev


## 4. Load QGIS-Prepared Flood CSV

**Purpose:** Load the QGIS-prepared barangay flood-area table retained after internal consistency/data-quality checks that was already produced separately in QGIS.

This notebook intentionally **does not** load shapefiles, repair geometries, reproject layers, intersect flood polygons, or convert shapefiles to CSV. Those spatial operations are treated as completed upstream.

In [5]:
def standardize_flood_columns(df: pd.DataFrame) -> pd.DataFrame:
    # Standardize common QGIS/export column variants to the names used downstream.
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    lower_map = {c: str(c).lower().strip().replace(" ", "_") for c in df.columns}
    df = df.rename(columns=lower_map)

    rename_candidates = {
        "barangay": ["barangay", "brgy", "brgy_name", "barangay_name", "adm4_name"],
        "brgy_area_sqm": ["brgy_area_sqm", "barangay_area_sqm", "area_sqm", "total_area_sqm", "shape_area"],
        "flood_area_5yr_sqm": ["flood_area_5yr_sqm", "flood_5yr_sqm", "flood_area_5yr", "area_5yr_sqm"],
        "flood_pct_5yr": ["flood_pct_5yr", "flood_5yr_pct", "pct_5yr", "flood_percent_5yr"],
        "flood_area_25yr_sqm": ["flood_area_25yr_sqm", "flood_25yr_sqm", "flood_area_25yr", "area_25yr_sqm"],
        "flood_pct_25yr": ["flood_pct_25yr", "flood_25yr_pct", "pct_25yr", "flood_percent_25yr"],
        "flood_area_100yr_sqm": ["flood_area_100yr_sqm", "flood_100yr_sqm", "flood_area_100yr", "area_100yr_sqm"],
        "flood_pct_100yr": ["flood_pct_100yr", "flood_100yr_pct", "pct_100yr", "flood_percent_100yr"],
    }

    for target, candidates in rename_candidates.items():
        if target not in df.columns:
            match = next((c for c in candidates if c in df.columns), None)
            if match:
                df = df.rename(columns={match: target})

    required = [
        "barangay", "brgy_area_sqm",
        "flood_area_5yr_sqm", "flood_pct_5yr",
        "flood_area_25yr_sqm", "flood_pct_25yr",
    ]

    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Flood CSV is missing required columns: {missing}. Available columns: {list(df.columns)}")

    df = safe_numeric(df, [c for c in df.columns if any(k in c for k in ["area", "pct", "sqm"])] )
    df["barangay"] = df["barangay"].astype(str).str.strip()
    df["barangay_no"] = df["barangay"].apply(extract_barangay_number)

    if KEEP_ONLY_NUMBERED_BARANGAYS:
        before = len(df)
        df = df[df["barangay"].apply(is_numbered_barangay)].copy()
        print(f"Numbered barangay filter: {before} -> {len(df)}")

    df = df.drop_duplicates(subset=["barangay"]).copy()
    df["Return_Period_Valid_5_25"] = df["flood_pct_25yr"] >= df["flood_pct_5yr"]
    df["flag_25yr_less_than_5yr"] = ~df["Return_Period_Valid_5_25"]
    return df

if not FLOOD_CSV.exists():
    raise FileNotFoundError(f"Required QGIS-prepared flood CSV not found: {FLOOD_CSV}")

flood_spatial_df = standardize_flood_columns(pd.read_csv(FLOOD_CSV))
assert_exact_barangay_count(flood_spatial_df, stage="qgis_flood_csv_load")

print("Loaded QGIS-prepared flood CSV:", FLOOD_CSV)
print("Rows:", len(flood_spatial_df))
print("5yr/25yr return-period anomalies:", int(flood_spatial_df["flag_25yr_less_than_5yr"].sum()))
display(flood_spatial_df.head())

Numbered barangay filter: 897 -> 897
[qgis_flood_csv_load] Barangay count validated: 897
Loaded QGIS-prepared flood CSV: MetroManila_Barangay_Flood_Area_Validated.csv
Rows: 897
5yr/25yr return-period anomalies: 2


,barangay,brgy_area_sqm,flood_area_5yr_sqm,flood_pct_5yr,flood_area_25yr_sqm,flood_pct_25yr,flood_area_100yr_sqm,flood_pct_100yr,flag_25yr_less_than_5yr,flag_100yr_less_than_25yr,flag_100yr_less_than_5yr,barangay_no,Return_Period_Valid_5_25
0,Barangay 1,49547.490522,8264.845530,16.680654,21069.565046,42.523980,24194.317057,48.830560,False,False,False,1,True
1,Barangay 10,6057.586255,5544.134717,91.523826,5999.129222,99.034978,6033.964192,99.610042,False,False,False,10,True
2,Barangay 100,29349.500168,8985.706920,30.616218,14348.687691,48.889036,15223.078876,51.868273,False,False,False,100,True
3,Barangay 101,103997.369545,37525.242932,36.082877,57378.997256,55.173508,61606.839736,59.238844,False,False,False,101,True
4,Barangay 102,23899.675836,4506.429582,18.855610,14200.178503,59.415779,20019.915620,83.766473,False,False,False,102,True


## 5. Validate and Copy Prepared Flood CSV

**Purpose:** Validate the already-prepared flood CSV and copy it to the output directory for reproducibility. No shapefile conversion is performed.

In [6]:
export_cols = [
    "barangay", "brgy_area_sqm",
    "flood_area_5yr_sqm", "flood_pct_5yr",
    "flood_area_25yr_sqm", "flood_pct_25yr",
    "Return_Period_Valid_5_25", "flag_25yr_less_than_5yr", "barangay_no",
]
export_cols = [c for c in export_cols if c in flood_spatial_df.columns]
flood_spatial_df[export_cols].to_csv(OUT_FLOOD_CSV, index=False)
print(f"Copied validated QGIS flood CSV to: {OUT_FLOOD_CSV}")
print("Rows:", len(flood_spatial_df))
display(flood_spatial_df[export_cols].head())


Copied validated QGIS flood CSV to: outputs/MetroManila_Barangay_Flood_Area_Validated.csv
Rows: 897


,barangay,brgy_area_sqm,flood_area_5yr_sqm,flood_pct_5yr,flood_area_25yr_sqm,flood_pct_25yr,Return_Period_Valid_5_25,flag_25yr_less_than_5yr,barangay_no
0,Barangay 1,49547.490522,8264.845530,16.680654,21069.565046,42.523980,True,False,1
1,Barangay 10,6057.586255,5544.134717,91.523826,5999.129222,99.034978,True,False,10
2,Barangay 100,29349.500168,8985.706920,30.616218,14348.687691,48.889036,True,False,100
3,Barangay 101,103997.369545,37525.242932,36.082877,57378.997256,55.173508,True,False,101
4,Barangay 102,23899.675836,4506.429582,18.855610,14200.178503,59.415779,True,False,102


## 6. Loading of PSA 2020 and PSA 2024 population datasets

**Purpose:** Load official PSA population snapshots for both census/reference years used in the thesis.

**Input:**
- PSA 2024: `population-dataset-2024.csv`
- PSA 2020: `population-dataset-2020.xlsx`

**Output:** Clean population tables with `Population_2020` and `Population_2024`.

**Why both years are integrated:** Population Change 2020–2024 (%) requires both snapshots. Using only one year would omit the demographic-change vulnerability indicator required by the updated methodology.


In [7]:
def load_psa_2024(csv_path):
    """
    Load PSA 2024 barangay population data.

    Supports:
    1. Headerless two-column CSV:
       Barangay, Population
    2. CSV with named barangay/population columns.
    """

    csv_path = Path(csv_path)

    if not csv_path.exists():
        raise FileNotFoundError(f"PSA 2024 file not found: {csv_path}")

    # Try headered read first
    try:
        df_raw = pd.read_csv(csv_path)
        df_raw.columns = [str(c).strip() for c in df_raw.columns]
    except Exception:
        df_raw = pd.DataFrame()

    barangay_candidates = [
        "Barangay", "barangay", "Barangay Name", "brgy_name",
        "adm4_name", "ADM4_NAME"
    ]

    pop_candidates = [
        "Population_2024", "population_2024", "Pop_2024",
        "Population", "population", "PSA_2024"
    ]

    barangay_col = next((c for c in barangay_candidates if c in df_raw.columns), None)
    pop_col = next((c for c in pop_candidates if c in df_raw.columns), None)

    # If headered detection fails, read as headerless two-column CSV
    if barangay_col is None or pop_col is None:
        df_raw = pd.read_csv(csv_path, header=None)
        df_raw = df_raw.dropna(how="all").dropna(axis=1, how="all")

        if df_raw.shape[1] < 2:
            raise ValueError(f"Could not detect barangay/population columns in {csv_path}")

        df = df_raw.iloc[:, :2].copy()
        df.columns = ["Barangay", "Population_2024"]

    else:
        df = df_raw[[barangay_col, pop_col]].copy()
        df.columns = ["Barangay", "Population_2024"]

    df["Barangay"] = df["Barangay"].astype(str).str.strip()

    df["Population_2024"] = (
        df["Population_2024"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    df["Population_2024"] = pd.to_numeric(df["Population_2024"], errors="coerce")

    # Keep valid barangay rows only
    df = df[
        df["Barangay"].str.contains(r"Barangay\s+\d+", case=False, na=False)
    ].copy()

    df = df[df["Population_2024"].notna()].copy()

    df["barangay_key_strict"] = df["Barangay"].apply(clean_barangay_name_strict)

    # Remove duplicate barangay records if present
    df = df.drop_duplicates("barangay_key_strict", keep="first").copy()

    # Sort
    df["Barangay_No"] = df["Barangay"].apply(extract_number)
    df = df.sort_values(["Barangay_No", "Barangay"]).drop(columns=["Barangay_No"]).reset_index(drop=True)

    print(f"Loaded PSA 2024 population: {len(df)} barangays")
    print("PSA 2024 total population:", int(df["Population_2024"].sum()))

    return df[["Barangay", "Population_2024", "barangay_key_strict"]]

def load_psa_2020(xlsx_path, base_csv=None):
    """
    Load PSA 2020 barangay population data.

    Supports:
    1. Simple two-column Excel layout:
       Barangay | Population
    2. Headerless two-column Excel layout.
    3. Multi-sheet PSA-style files with barangay and population columns.
    4. Optional fallback to base admin CSV if it contains 2020 population.
    """

    xlsx_path = Path(xlsx_path)

    if not xlsx_path.exists():
        raise FileNotFoundError(f"PSA 2020 file not found: {xlsx_path}")

    frames = []

    # Read all sheets with no assumed header first
    sheets = pd.read_excel(xlsx_path, sheet_name=None, header=None)

    for sheet_name, raw in sheets.items():
        tmp = raw.copy()
        tmp = tmp.dropna(how="all").dropna(axis=1, how="all")

        if tmp.empty or tmp.shape[1] < 2:
            continue

        # Flatten to first two useful columns
        tmp = tmp.iloc[:, :2].copy()
        tmp.columns = ["Barangay", "Population_2020"]

        tmp["Barangay"] = tmp["Barangay"].astype(str).str.strip()
        tmp["Population_2020"] = (
            tmp["Population_2020"]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.strip()
        )
        tmp["Population_2020"] = pd.to_numeric(tmp["Population_2020"], errors="coerce")

        # Keep rows that look like Manila barangay rows
        tmp = tmp[
            tmp["Barangay"].str.contains(r"Barangay\s+\d+", case=False, na=False)
        ].copy()

        tmp = tmp[tmp["Population_2020"].notna()].copy()

        if len(tmp) > 0:
            frames.append(tmp)

    if not frames:
        # Optional fallback to base admin file
        if base_csv is not None and Path(base_csv).exists():
            base = pd.read_csv(base_csv)
            base.columns = [str(c).strip() for c in base.columns]

            brgy_col = next(
                (
                    c for c in [
                        "Barangay", "barangay", "adm4_name", "ADM4_NAME",
                        "brgy_name", "Brgy_Name", "NAME_4"
                    ]
                    if c in base.columns
                ),
                None
            )

            pop_col = next(
                (
                    c for c in [
                        "Population_2020", "Pop_2020", "bgypop2020",
                        "bgy_pop2020", "pop2020", "PSA_2020"
                    ]
                    if c in base.columns
                ),
                None
            )

            if brgy_col and pop_col:
                df = base[[brgy_col, pop_col]].copy()
                df.columns = ["Barangay", "Population_2020"]
                df["Population_2020"] = (
                    df["Population_2020"]
                    .astype(str)
                    .str.replace(",", "", regex=False)
                    .str.strip()
                )
                df["Population_2020"] = pd.to_numeric(df["Population_2020"], errors="coerce")
                df = df[df["Population_2020"].notna()].copy()
                df["barangay_key_strict"] = df["Barangay"].apply(clean_barangay_name_strict)
                df = df.drop_duplicates("barangay_key_strict")
                print(f"Loaded PSA 2020 fallback from base admin CSV: {len(df)} rows")
                return df[["Barangay", "Population_2020", "barangay_key_strict"]]

        raise ValueError(f"Could not detect barangay/population columns in {xlsx_path}")

    df = pd.concat(frames, ignore_index=True)

    df["Barangay"] = df["Barangay"].astype(str).str.strip()
    df["barangay_key_strict"] = df["Barangay"].apply(clean_barangay_name_strict)

    # Remove duplicate barangay records if any
    df = df.drop_duplicates("barangay_key_strict", keep="first").copy()

    # Final sort
    df["Barangay_No"] = df["Barangay"].apply(extract_number)
    df = df.sort_values(["Barangay_No", "Barangay"]).drop(columns=["Barangay_No"]).reset_index(drop=True)

    print(f"Loaded PSA 2020 population: {len(df)} barangays")
    print("PSA 2020 total population:", int(df["Population_2020"].sum()))

    return df[["Barangay", "Population_2020", "barangay_key_strict"]]


df_pop_2024 = load_psa_2024(PSA_2024_CSV)
df_pop_2020 = load_psa_2020(PSA_2020_XLSX, BASE_ADMIN_CSV)


Loaded PSA 2024 population: 897 barangays
PSA 2024 total population: 1902590
Loaded PSA 2020 population: 897 barangays
PSA 2020 total population: 1846513


## 7. Barangay Name/Key Standardization

**Purpose:** Create consistent merge keys across flood, population, and administrative tables.

**Input:** Raw barangay name strings from each source.

**Output:** `barangay_clean` and `barangay_key_strict` fields.

**Why needed:** Sources use inconsistent spacing, casing, and hyphenation (e.g., `Brgy. 202-A` vs `Barangay 202A`). Standardization is required for a complete 897-barangay match.


In [8]:
def add_merge_keys(df: pd.DataFrame, barangay_col: str = "Barangay") -> pd.DataFrame:
    df = df.copy()
    df[barangay_col] = df[barangay_col].astype(str).str.strip()
    df["barangay_clean"] = df[barangay_col].apply(clean_barangay_name)
    df["barangay_key_strict"] = df[barangay_col].apply(clean_barangay_name_strict)
    return df


df_pop_2024 = add_merge_keys(df_pop_2024, "Barangay")
df_pop_2020 = add_merge_keys(df_pop_2020, "Barangay")

df_flood = flood_spatial_df.rename(columns={
    "barangay": "Barangay",
    "brgy_area_sqm": "Barangay_Area_Sqm",
    "flood_area_5yr_sqm": "Flood_Area_5yr_Sqm",
    "flood_pct_5yr": "Flood_PCT_5yr",
    "flood_area_25yr_sqm": "Flood_Area_25yr_Sqm",
    "flood_pct_25yr": "Flood_PCT_25yr",
    "barangay_no": "Barangay_No",
}).copy()
df_flood = add_merge_keys(df_flood, "Barangay")

print("Merge keys created for population and flood tables.")
print("2024 keys sample:", df_pop_2024[["Barangay", "barangay_key_strict"]].head(3).to_dict("records"))
print("Flood keys sample:", df_flood[["Barangay", "barangay_key_strict"]].head(3).to_dict("records"))


Merge keys created for population and flood tables.
2024 keys sample: [{'Barangay': 'Barangay 1', 'barangay_key_strict': 'BARANGAY1'}, {'Barangay': 'Barangay 2', 'barangay_key_strict': 'BARANGAY2'}, {'Barangay': 'Barangay 3', 'barangay_key_strict': 'BARANGAY3'}]
Flood keys sample: [{'Barangay': 'Barangay 1', 'barangay_key_strict': 'BARANGAY1'}, {'Barangay': 'Barangay 10', 'barangay_key_strict': 'BARANGAY10'}, {'Barangay': 'Barangay 100', 'barangay_key_strict': 'BARANGAY100'}]


## 8. Population Dataset Validation and 897-Barangay Matching

**Purpose:** Merge PSA 2020 and PSA 2024 onto the flood barangay universe and validate complete coverage.

**Input:** Flood table (897 barangays) + PSA 2020 + PSA 2024.

**Output:** Matched barangay table with both population years; citywide total checks.

**Why needed:** The thesis uses total enumeration of Manila's 897 barangays. Missing population values would bias exposure and vulnerability scores.


In [9]:
df = df_flood.merge(
    df_pop_2024[["barangay_key_strict", "Population_2024"]],
    on="barangay_key_strict",
    how="left",
)
df = df.merge(
    df_pop_2020[["barangay_key_strict", "Population_2020"]],
    on="barangay_key_strict",
    how="left",
)

print("After population merges:", len(df))
print("Missing Population_2024:", int(df["Population_2024"].isna().sum()))
print("Missing Population_2020:", int(df["Population_2020"].isna().sum()))

missing_2024 = df.loc[df["Population_2024"].isna(), ["Barangay", "barangay_key_strict"]]
missing_2020 = df.loc[df["Population_2020"].isna(), ["Barangay", "barangay_key_strict"]]
if len(missing_2024):
    print("Barangays missing PSA 2024:")
    print(missing_2024.head(20).to_string(index=False))
if len(missing_2020):
    print("Barangays missing PSA 2020:")
    print(missing_2020.head(20).to_string(index=False))

if df["Population_2024"].isna().any() or df["Population_2020"].isna().any():
    raise ValueError(
        "Population integration incomplete: one or more barangays lack Population_2020 or Population_2024."
    )

assert_exact_barangay_count(df, stage="population_match")

total_2020 = float(df["Population_2020"].sum())
total_2024 = float(df["Population_2024"].sum())
change_abs = total_2024 - total_2020
change_pct = (change_abs / total_2020) * 100 if total_2020 else np.nan

print(f"\nCitywide Population_2020: {total_2020:,.0f} (expected {EXPECTED_PSA_2020_TOTAL:,})")
print(f"Citywide Population_2024: {total_2024:,.0f} (expected {EXPECTED_PSA_2024_TOTAL:,})")
print(f"Absolute change: {change_abs:,.0f} (expected {EXPECTED_POP_CHANGE_ABS:,})")
print(f"Percentage change: {change_pct:.4f}% (expected ~{EXPECTED_POP_CHANGE_PCT}%)")

if abs(total_2020 - EXPECTED_PSA_2020_TOTAL) > 1:
    print("WARNING: PSA 2020 citywide total differs from expected reference.")
if abs(total_2024 - EXPECTED_PSA_2024_TOTAL) > 1:
    print("WARNING: PSA 2024 citywide total differs from expected reference.")
if abs(change_abs - EXPECTED_POP_CHANGE_ABS) > 1:
    print("WARNING: Absolute population change differs from expected reference.")


After population merges: 897
Missing Population_2024: 0
Missing Population_2020: 0
[population_match] Barangay count validated: 897

Citywide Population_2020: 1,846,513 (expected 1,846,513)
Citywide Population_2024: 1,902,590 (expected 1,902,590)
Absolute change: 56,077 (expected 56,077)
Percentage change: 3.0369% (expected ~3.04%)


## 9. Integration of Flood, Population, Area, and Required Elevation Data

**Purpose:** Attach required QGIS-derived elevation statistics to the matched flood/population data.

**Input:** Matched flood-population table + required QGIS Zonal Statistics elevation CSV.

**Output:** Integrated analytical dataframe for feature engineering.

**Why Needed:** The vulnerability computation requires complete elevation coverage.

**Elevation Rule:** `Elevation_Mean` must be present for all 897 barangays. If any barangay lacks mean elevation after the merge, the notebook raises an error.


In [10]:
# Merge required QGIS elevation data

if not ELEVATION_CSV.exists():
    raise FileNotFoundError(
        f"Required elevation CSV not found: {ELEVATION_CSV}. "
        "Provide the QGIS Zonal Statistics output file."
    )

# Load and standardize QGIS elevation table
elevation_df = pd.read_csv(ELEVATION_CSV)
elevation_df = standardize_elevation_columns(elevation_df)

validate_required_columns(
    elevation_df,
    ["Barangay", "Elevation_Mean"],
    dataset_name="Elevation CSV"
)

# Build strict merge key
elevation_df["barangay_key_strict"] = elevation_df["Barangay"].apply(clean_barangay_name_strict)

# Keep only useful elevation columns that exist
elevation_keep_cols = [
    "barangay_key_strict",
    "Barangay",
    "Elevation_Count",
    "Elevation_Mean",
    "Elevation_Median",
    "Elevation_Min",
    "Elevation_Max",
]

elevation_keep_cols = [c for c in elevation_keep_cols if c in elevation_df.columns]
elevation_df = elevation_df[elevation_keep_cols].copy()

# Convert elevation values to numeric
numeric_elev_cols = [
    "Elevation_Count",
    "Elevation_Mean",
    "Elevation_Median",
    "Elevation_Min",
    "Elevation_Max",
]
elevation_df = safe_numeric(elevation_df, numeric_elev_cols)

# Check duplicate barangay keys in elevation source
duplicate_elev = elevation_df[elevation_df["barangay_key_strict"].duplicated(keep=False)]
if len(duplicate_elev) > 0:
    print("Duplicate barangay keys found in elevation file:")
    display(duplicate_elev.sort_values("barangay_key_strict"))
    raise ValueError("Duplicate barangay keys found in elevation data. Clean the elevation CSV before merging.")

# Optional diagnostic: show non-standard/non-numbered entries
non_numbered_elev = elevation_df[
    ~elevation_df["Barangay"].astype(str).apply(is_numbered_barangay)
].copy()

if len(non_numbered_elev) > 0:
    print("Non-numbered rows detected in elevation file. These will only merge if present in the main dataset:")
    display(non_numbered_elev[["Barangay"]].head(20))

# Merge elevation into main dataframe
df = df.merge(
    elevation_df.drop(columns=["Barangay"], errors="ignore"),
    on="barangay_key_strict",
    how="left"
)

print("Elevation merged from:", ELEVATION_CSV)
print("Dataset shape after elevation merge:", df.shape)
print("Missing Elevation_Mean:", int(df["Elevation_Mean"].isna().sum()))

# Validate barangay count remains 897
assert_exact_barangay_count(df, stage="after_elevation_merge")

# Elevation is required for final methodology
missing_elev_count = int(df["Elevation_Mean"].isna().sum())
if missing_elev_count > 0:
    missing_elev = df.loc[
        df["Elevation_Mean"].isna(),
        ["Barangay", "barangay_key_strict"]
    ].copy()

    print("Barangays missing elevation:")
    display(missing_elev)

    raise ValueError(
        f"{missing_elev_count} barangays are missing Elevation_Mean after merge. "
        "Check barangay naming/key matching between the main dataset and the QGIS elevation CSV."
    )

print("Elevation validation passed: all 897 barangays have Elevation_Mean.")

# Quick elevation summary for audit
display(
    df[["Elevation_Mean", "Elevation_Median", "Elevation_Min", "Elevation_Max"]]
    .describe()
    .T
)

Elevation merged from: manila_barangay_elevation.csv
Dataset shape after elevation merge: (897, 22)
Missing Elevation_Mean: 0
[after_elevation_merge] Barangay count validated: 897
Elevation validation passed: all 897 barangays have Elevation_Mean.


,count,mean,std,min,25%,50%,75%,max
Elevation_Mean,897.0,2.328222,1.200392,0.0,1.691874,2.0,3.0,9.141101
Elevation_Median,897.0,2.347269,1.192927,0.0,2.000000,2.0,3.0,9.000000
Elevation_Min,897.0,1.746934,1.180928,0.0,1.000000,2.0,2.0,8.000000
Elevation_Max,897.0,2.926421,1.850779,0.0,2.000000,3.0,3.0,30.000000


## 10. Feature Engineering

**Purpose:** Compute thesis indicators for hazard escalation, exposure, density, and population change.

**Key Definitions (Updated Methodology):**
- `Population_Change_2020_2024 = Population_2024 - Population_2020`
- `Population_Change_2020_2024_Pct = ((Population_2024 - Population_2020) / Population_2020) * 100` — **not annualized**
- `Barangay_Area_Hectares = Barangay_Area_Sqm / 10000`
- `Population_Density_per_Hectare = Population_2024 / Barangay_Area_Hectares`
- Affected population uses **PSA 2024** only
- `Flood_Escalation_5to25` and `Flood_Severity_Increase_5to25_Sqm` flagged if negative

**Why Population Change (not growth rate):** The indicator is a four-year observed change between official PSA snapshots, not an annualized demographic rate.

**Why Density Uses 2024:** Exposure and current vulnerability should reflect the more recent population snapshot.

**Why Relative Population Density is removed:** The updated vulnerability specification uses absolute density, population change, and elevation only. Relative density is excluded from vulnerability and from the final ML feature list.


In [11]:
df = safe_numeric(
    df,
    [
        "Population_2020", "Population_2024", "Barangay_Area_Sqm",
        "Flood_Area_5yr_Sqm", "Flood_PCT_5yr", "Flood_Area_25yr_Sqm", "Flood_PCT_25yr",
        "bgyarea_sqkm", "Elevation_Mean", "Elevation_Median", "Elevation_Min", "Elevation_Max", "Elevation_Count",
    ],
)

df["Population_Change_2020_2024"] = df["Population_2024"] - df["Population_2020"]
df["Population_Change_2020_2024_Pct"] = np.where(
    df["Population_2020"] > 0,
    (df["Population_Change_2020_2024"] / df["Population_2020"]) * 100,
    np.nan,
)

df["Barangay_Area_Hectares"] = df["Barangay_Area_Sqm"] / 10_000
if (df["Barangay_Area_Hectares"] <= 0).any():
    raise ValueError("Non-positive barangay area in hectares detected.")
df["Population_Density_per_Hectare"] = df["Population_2024"] / df["Barangay_Area_Hectares"]

df["Affected_Population_5yr"] = df["Population_2024"] * (df["Flood_PCT_5yr"] / 100)
df["Affected_Population_25yr"] = df["Population_2024"] * (df["Flood_PCT_25yr"] / 100)
df["Affected_Population_Increase_5to25"] = (
    df["Affected_Population_25yr"] - df["Affected_Population_5yr"]
)

df["Flood_Escalation_5to25"] = df["Flood_PCT_25yr"] - df["Flood_PCT_5yr"]
df["Flood_Severity_Increase_5to25_Sqm"] = df["Flood_Area_25yr_Sqm"] - df["Flood_Area_5yr_Sqm"]

neg_esc = df["Flood_Escalation_5to25"] < -EPS
neg_sev = df["Flood_Severity_Increase_5to25_Sqm"] < -EPS
if neg_esc.any() or neg_sev.any():
    print(
        f"WARNING: Negative escalation/severity detected "
        f"(escalation={int(neg_esc.sum())}, severity_sqm={int(neg_sev.sum())})."
    )
    print(
        df.loc[
            neg_esc | neg_sev,
            [
                "Barangay", "Flood_PCT_5yr", "Flood_PCT_25yr",
                "Flood_Escalation_5to25", "Flood_Severity_Increase_5to25_Sqm",
            ],
        ].head(20)
    )

if "bgyarea_sqkm" in df.columns:
    df["Base_Area_Sqm"] = df["bgyarea_sqkm"] * 1_000_000
    df["Area_Difference_Sqm"] = df["Barangay_Area_Sqm"] - df["Base_Area_Sqm"]
    df["Area_Difference_Pct"] = np.where(
        df["Base_Area_Sqm"] > 0,
        (df["Area_Difference_Sqm"] / df["Base_Area_Sqm"]) * 100,
        np.nan,
    )

if "city_name" in df.columns:
    df["District_Avg_Flood_PCT_5yr"] = df.groupby("city_name")["Flood_PCT_5yr"].transform("mean")
    df["District_Avg_Flood_PCT_25yr"] = df.groupby("city_name")["Flood_PCT_25yr"].transform("mean")
    df["District_Avg_Population_Density_per_Hectare"] = (
        df.groupby("city_name")["Population_Density_per_Hectare"].transform("mean")
    )
    df["Relative_Flood_Risk_25yr_vs_District"] = (
        df["Flood_PCT_25yr"] - df["District_Avg_Flood_PCT_25yr"]
    )
    # NOTE: Relative Population Density is intentionally NOT computed for vulnerability/ML.

df["Sort_Key"] = df["Barangay"].apply(extract_number)
df = df.sort_values(["Sort_Key", "Barangay"]).drop(columns=["Sort_Key"]).reset_index(drop=True)

print("Feature Engineering complete.")
print(
    df[[
        "Barangay", "Population_2020", "Population_2024", "Population_Change_2020_2024_Pct",
        "Population_Density_per_Hectare", "Flood_Escalation_5to25",
        "Affected_Population_5yr", "Affected_Population_25yr",
        "Elevation_Mean", "Elevation_Median",
    ]].head()
)



Feature Engineering complete.
     Barangay  Population_2020  Population_2024  Population_Change_2020_2024_Pct  Population_Density_per_Hectare  Flood_Escalation_5to25  \
0  Barangay 1             2995           3119.0                         4.140234                      629.497068               25.843326   
1  Barangay 2             1887           1666.0                       -11.711712                      439.740638               30.180490   
2  Barangay 3             1341           1232.0                        -8.128262                      316.379617               25.033508   
3  Barangay 4             1691           1717.0                         1.537552                      427.181509               43.834370   
4  Barangay 5             1288           1321.0                         2.562112                      533.913235               43.535626   

   Affected_Population_5yr  Affected_Population_25yr  Elevation_Mean  Elevation_Median  
0               520.269603              

## 11. Export of Enriched Analytical Dataset

**Purpose:** Save the full integrated barangay table for documentation and downstream DPI/ML.

**Input:** Feature-engineered dataframe.

**Output:**
- `manila-city-flood-population_enriched_NEW.csv`
- `manila-city-flood-population_enriched_NEW.xlsx`

**Why Needed:** Provides a human-readable and machine-readable archive of all analytical fields before index construction.


In [12]:
assert_exact_barangay_count(df, stage="enriched_export")

geom_cols = [c for c in df.columns if str(c).lower() in {"geometry", "geom"}]
if geom_cols:
    df = df.drop(columns=geom_cols)

df.to_csv(OUT_ENRICHED_CSV, index=False)
try:
    df.to_excel(OUT_ENRICHED_XLSX, index=False)
    print(f"Exported Excel: {OUT_ENRICHED_XLSX}")
except Exception as exc:
    print(f"Excel export skipped ({exc})")

print(f"Exported enriched CSV: {OUT_ENRICHED_CSV}")
print("Final enriched shape:", df.shape)
print("Missing value summary (top 15):")
print(df.isna().sum().sort_values(ascending=False).head(15))


[enriched_export] Barangay count validated: 897
Exported Excel: outputs/manila-city-flood-population_enriched_NEW.xlsx
Exported enriched CSV: outputs/manila-city-flood-population_enriched_NEW.csv
Final enriched shape: (897, 31)
Missing value summary (top 15):
Barangay                     0
Barangay_Area_Sqm            0
Flood_Area_5yr_Sqm           0
Flood_PCT_5yr                0
Flood_Area_25yr_Sqm          0
Flood_PCT_25yr               0
flood_area_100yr_sqm         0
flood_pct_100yr              0
flag_25yr_less_than_5yr      0
flag_100yr_less_than_25yr    0
flag_100yr_less_than_5yr     0
Barangay_No                  0
Return_Period_Valid_5_25     0
barangay_clean               0
barangay_key_strict          0
dtype: int64


## 12. Preparation of ML-Ready Dataset Scaffold

**Purpose:** Identify identifier columns versus candidate analytic fields ahead of Hazard/DPI computation.

**Input:** Enriched dataset.

**Output:** Working dataframe `df_ml` for index and modeling stages.

**Why Needed:** Separates identifiers from numeric indicators and prepares a clean table for leakage-controlled modeling.


In [13]:
df_ml = df.copy()
df_ml.columns = df_ml.columns.astype(str).str.strip()

id_cols = keep_existing(
    [
        "Barangay", "Barangay_No", "psgc_10d", "city_name", "brgy_code",
        "Population_2020", "Population_2024",
    ],
    df_ml,
)
print("Identifier / reference columns:", id_cols)
print("Working ML dataframe shape:", df_ml.shape)
assert_exact_barangay_count(df_ml, stage="ml_scaffold")


Identifier / reference columns: ['Barangay', 'Barangay_No', 'Population_2020', 'Population_2024']
Working ML dataframe shape: (897, 31)
[ml_scaffold] Barangay count validated: 897


## 13. Hazard Computation

**Purpose:** Compute the Hazard Component from retained 5-year and 25-year indicators only.

**Indicators (equal-weighted after min-max normalization):**
- `Flood_PCT_5yr`
- `Flood_PCT_25yr`
- `Flood_Escalation_5to25`
- `Flood_Severity_Increase_5to25_Sqm`

**Why the 100-year layer is excluded:** Return-period consistency issues in the 100-year source layer would distort Hazard and downstream DPI classes.


In [14]:
hazard_inputs = [
    "Flood_PCT_5yr",
    "Flood_PCT_25yr",
    "Flood_Escalation_5to25",
    "Flood_Severity_Increase_5to25_Sqm",
]
missing = [c for c in hazard_inputs if c not in df_ml.columns]
if missing:
    raise ValueError(f"Missing Hazard inputs: {missing}")

df_ml = safe_numeric(df_ml, hazard_inputs)
df_ml["norm_Flood_PCT_5yr"] = minmax_normalize(df_ml["Flood_PCT_5yr"])
df_ml["norm_Flood_PCT_25yr"] = minmax_normalize(df_ml["Flood_PCT_25yr"])
df_ml["norm_Flood_Escalation_5to25"] = minmax_normalize(df_ml["Flood_Escalation_5to25"])
df_ml["norm_Flood_Severity_Increase_5to25_Sqm"] = minmax_normalize(
    df_ml["Flood_Severity_Increase_5to25_Sqm"]
)

df_ml["Hazard"] = df_ml[[
    "norm_Flood_PCT_5yr",
    "norm_Flood_PCT_25yr",
    "norm_Flood_Escalation_5to25",
    "norm_Flood_Severity_Increase_5to25_Sqm",
]].mean(axis=1)

print("Hazard computed. Summary:")
print(df_ml["Hazard"].describe())


Hazard computed. Summary:
count    8.970000e+02
mean     4.205806e-01
std      1.474927e-01
min      1.754438e-12
25%      3.562610e-01
50%      5.000000e-01
75%      5.105478e-01
max      6.404339e-01
Name: Hazard, dtype: float64


## 14. Exposure Score Computation

**Purpose:** Quantify population potentially affected under 5-year and 25-year flood scenarios using PSA 2024.

**Indicators (Equal-Weighted after Normalization):**
- `Affected_Population_5yr`
- `Affected_Population_25yr`
- `Affected_Population_Increase_5to25`

`Exposure = mean(normalized exposure indicators)`

**Why Separate from Hazard:** Exposure captures **who is affected**, not only how much land is flooded.


In [15]:
exposure_inputs = [
    "Affected_Population_5yr",
    "Affected_Population_25yr",
    "Affected_Population_Increase_5to25",
]
missing = [c for c in exposure_inputs if c not in df_ml.columns]
if missing:
    raise ValueError(f"Missing Exposure inputs: {missing}")

df_ml = safe_numeric(df_ml, exposure_inputs)
df_ml["norm_Affected_Population_5yr"] = minmax_normalize(df_ml["Affected_Population_5yr"])
df_ml["norm_Affected_Population_25yr"] = minmax_normalize(df_ml["Affected_Population_25yr"])
df_ml["norm_Affected_Population_Increase_5to25"] = minmax_normalize(
    df_ml["Affected_Population_Increase_5to25"]
)
df_ml["Exposure"] = df_ml[[
    "norm_Affected_Population_5yr",
    "norm_Affected_Population_25yr",
    "norm_Affected_Population_Increase_5to25",
]].mean(axis=1)

print("Exposure computed. Summary:")
print(df_ml["Exposure"].describe())


Exposure computed. Summary:
count    8.970000e+02
mean     8.724946e-02
std      8.365317e-02
min      5.524718e-13
25%      3.956010e-02
50%      6.894792e-02
75%      1.065094e-01
max      8.478075e-01
Name: Exposure, dtype: float64


## 15. Vulnerability Score Computation

**Purpose:** Compute demographic/physical susceptibility using the final three-indicator specification.

**Indicators:**
1. `Population_Density_per_Hectare` — min-max normalized
2. `Population_Change_2020_2024_Pct` — min-max normalized
3. `Elevation_Mean` — reverse min-max normalized

The vulnerability score is:

$$Vulnerability_i = \frac{NormDensity_i + NormPopChange_i + ReverseNormElevation_i}{3}$$

**Why Elevation is Reverse-Normalized:** Lower-lying barangays are generally more susceptible to inundation, so lower mean elevation receives a higher vulnerability contribution.


In [16]:
# Vulnerability score computation

vuln_inputs = [
    "Population_Density_per_Hectare",
    "Population_Change_2020_2024_Pct",
    "Elevation_Mean",
]

missing = [c for c in vuln_inputs if c not in df_ml.columns]
if missing:
    raise ValueError(f"Missing required vulnerability inputs: {missing}")

df_ml = safe_numeric(df_ml, vuln_inputs)

if df_ml["Elevation_Mean"].isna().sum() > 0:
    missing_elev = df_ml.loc[
        df_ml["Elevation_Mean"].isna(),
        ["Barangay", "barangay_key_strict"]
    ]
    print("Barangays missing Elevation_Mean:")
    display(missing_elev)

    raise ValueError(
        "Elevation_Mean is required for vulnerability computation, "
        "but some barangays have missing elevation."
    )

df_ml["norm_Population_Density_per_Hectare"] = minmax_normalize(
    df_ml["Population_Density_per_Hectare"]
)

df_ml["norm_Population_Change_2020_2024_Pct"] = minmax_normalize(
    df_ml["Population_Change_2020_2024_Pct"]
)

df_ml["norm_Elevation_reverse"] = minmax_reverse(
    df_ml["Elevation_Mean"]
)

vulnerability_components = [
    "norm_Population_Density_per_Hectare",
    "norm_Population_Change_2020_2024_Pct",
    "norm_Elevation_reverse",
]

df_ml["Vulnerability"] = df_ml[vulnerability_components].mean(axis=1)
df_ml["Vulnerability_Elevation_Included"] = True

print("Vulnerability components:", vulnerability_components)
print("Vulnerability includes required reverse-normalized Elevation_Mean.")
print(df_ml["Vulnerability"].describe())

display(
    df_ml[[
        "Barangay",
        "Population_Density_per_Hectare",
        "Population_Change_2020_2024_Pct",
        "Elevation_Mean",
        "norm_Elevation_reverse",
        "Vulnerability",
    ]].head()
)


Vulnerability components: ['norm_Population_Density_per_Hectare', 'norm_Population_Change_2020_2024_Pct', 'norm_Elevation_reverse']
Vulnerability includes required reverse-normalized Elevation_Mean.
count    897.000000
mean       0.341489
std        0.063581
min        0.048382
25%        0.308613
50%        0.336949
75%        0.372616
max        0.630506
Name: Vulnerability, dtype: float64


,Barangay,Population_Density_per_Hectare,Population_Change_2020_2024_Pct,Elevation_Mean,norm_Elevation_reverse,Vulnerability
0,Barangay 1,629.497068,4.140234,2.569270,0.718932,0.327883
1,Barangay 2,439.740638,-11.711712,2.726790,0.701700,0.298873
2,Barangay 3,316.379617,-8.128262,2.581515,0.717593,0.295507
3,Barangay 4,427.181509,1.537552,3.000000,0.671812,0.294023
4,Barangay 5,533.913235,2.562112,3.000000,0.671812,0.303438


## 16. Entropy Weighting

**Purpose:** Derive objective weights for the three high-level DPI components (Hazard, Exposure, Vulnerability).

**Input:** Component score columns.

**Output:** Entropy values, degree of diversification, and entropy weights (printed).

**Why Entropy Weighting Across Component Scores:** The thesis weights the three constructs by information content across barangays rather than assigning arbitrary fixed weights. Raw indicators are not entropy-weighted individually in the final method.


In [17]:
component_cols = ["Hazard", "Exposure", "Vulnerability"]
entropy_w, entropy_values, diversification_values = entropy_weights(df_ml[component_cols])

weights_table = pd.DataFrame({
    "Component": entropy_w.index,
    "Entropy_Value": entropy_values.values,
    "Diversification_Value": diversification_values.values,
    "Entropy_Weight": entropy_w.values,
})
print("Entropy-derived component weights:")
print(weights_table.to_string(index=False))
print("Weight sum:", float(entropy_w.sum()))
weights_table.to_csv(OUTPUT_DIR / "entropy_component_weights.csv", index=False)


Entropy-derived component weights:
    Component  Entropy_Value  Diversification_Value  Entropy_Weight
       Hazard       0.987678               0.012322        0.192859
     Exposure       0.952112               0.047888        0.749503
Vulnerability       0.996317               0.003683        0.057638
Weight sum: 1.0


## 17. DPI Computation and Scaling

**Purpose:** Combine weighted components into the Disaster Prioritization Index and rescale to 0–100.

`DPI = w_Hazard * Hazard + w_Exposure * Exposure + w_Vulnerability * Vulnerability`

`DPI_Scaled = minmax(DPI) * 100`


In [18]:
df_ml["DPI"] = (
    entropy_w["Hazard"] * df_ml["Hazard"]
    + entropy_w["Exposure"] * df_ml["Exposure"]
    + entropy_w["Vulnerability"] * df_ml["Vulnerability"]
)
df_ml["DPI_Scaled"] = minmax_normalize(df_ml["DPI"]) * 100

print("DPI computed and scaled to 0-100.")
print(df_ml[["Barangay", "Hazard", "Exposure", "Vulnerability", "DPI", "DPI_Scaled"]].head())
print(df_ml["DPI_Scaled"].describe())


DPI computed and scaled to 0-100.
     Barangay    Hazard  Exposure  Vulnerability       DPI  DPI_Scaled
0  Barangay 1  0.250762  0.083278       0.327883  0.129677   15.981407
1  Barangay 2  0.256200  0.046313       0.298873  0.101349   12.312496
2  Barangay 3  0.516438  0.070001       0.295507  0.169098   21.086870
3  Barangay 4  0.388906  0.072191       0.294023  0.146058   18.102935
4  Barangay 5  0.403388  0.059435       0.303438  0.139833   17.296712
count    897.000000
mean      20.710152
std        9.978904
min        0.000000
25%       16.248725
50%       20.122083
75%       24.816529
max      100.000000
Name: DPI_Scaled, dtype: float64


## 18. DPI Class Generation using Tertiles

**Purpose:** Create ordinal priority classes Low / Medium / High from scaled DPI using tertiles (`qcut`).

**Output Column:** `DPI_Risk_Class`

**Why DPI-derived Classes are ML Targets:** Externally observed flood-impact labels are not available in this dataset. Models learn to approximate the Disaster Prioritization Index classes for methodological comparison — **not** historical flood-event outcomes and **not** real-time flood prediction.

If duplicate bin edges occur, `duplicates='drop'` is used and the approach is reported.


In [19]:
try:
    df_ml["DPI_Risk_Class"] = pd.qcut(
        df_ml["DPI_Scaled"],
        q=3,
        labels=["Low", "Medium", "High"],
        duplicates="drop",
    )
    qcut_note = "Tertile classification via pd.qcut on DPI_Scaled (duplicates='drop')."
except ValueError as exc:
    ranks = df_ml["DPI_Scaled"].rank(method="first")
    df_ml["DPI_Risk_Class"] = pd.qcut(ranks, q=3, labels=["Low", "Medium", "High"])
    qcut_note = (
        f"qcut on raw DPI_Scaled failed ({exc}); used rank-based qcut to force three classes."
    )

print(qcut_note)
print("DPI_Risk_Class distribution:")
print(df_ml["DPI_Risk_Class"].value_counts().to_string())
print("Class proportions (%):")
print((df_ml["DPI_Risk_Class"].value_counts(normalize=True) * 100).round(2).to_string())


Tertile classification via pd.qcut on DPI_Scaled (duplicates='drop').
DPI_Risk_Class distribution:
DPI_Risk_Class
Low       299
Medium    299
High      299
Class proportions (%):
DPI_Risk_Class
Low       33.33
Medium    33.33
High      33.33


## 19. Direct-leakage-controlled ML Feature Selection

**Purpose:** Build the predictor matrix while excluding all target- and index-derived fields.

**Excluded (non-exhaustive reminder):** Hazard, Exposure, Vulnerability, DPI, DPI_Scaled, DPI_Risk_Class, normalized component intermediates, identifiers, validation flags, geometry, 100-year fields, Relative Population Density, Area_Difference_Pct (QC only).

**Recommended Predictors (included when present):**
Flood percentages/escalation/severity, Population_2024, affected population fields, density, population-change %, elevation, The manuscript specifies exactly 11 predictors; district-context fields are retained in the enriched dataset but excluded from the deployed comparison model.

**Important limitation:** DPI components and labels are excluded as direct inputs. The raw/engineered predictors still construct the DPI labels; the ML task reproduces an index, not independent real-world flood outcomes.


In [20]:
target_col = "DPI_Risk_Class"

candidate_features = [
    # Hazard raw/engineered predictors
    "Flood_PCT_5yr",
    "Flood_PCT_25yr",
    "Flood_Escalation_5to25",
    "Flood_Severity_Increase_5to25_Sqm",

    # Exposure raw/engineered predictors
    "Population_2024",
    "Affected_Population_5yr",
    "Affected_Population_25yr",
    "Affected_Population_Increase_5to25",

    # Vulnerability raw predictors
    "Population_Density_per_Hectare",
    "Population_Change_2020_2024_Pct",
    "Elevation_Mean",

    # District-context variables remain available elsewhere for diagnostics only.
    # Excluded here to match the 11 predictors specified in the thesis.

]

leakage_or_forbidden = {
    "Hazard", "Hazard_Score", "Hazard Score", "Hazard Score",
    "Exposure", "Exposure_Score", "Exposure Score",
    "Vulnerability", "Vulnerability_Score", "Vulnerability Score",
    "DPI", "DPI_Scaled", "DPI Score", "DPI Score 0to100",
    "DPI_Risk_Class", "DPI Risk Class",
    "Entropy_Weight", "Entropy_Value", "Diversification_Value",

    # Normalized/index intermediate variables
    "norm_Flood_PCT_5yr",
    "norm_Flood_PCT_25yr",
    "norm_Flood_Escalation_5to25",
    "norm_Flood_Severity_Increase_5to25_Sqm",
    "norm_Affected_Population_5yr",
    "norm_Affected_Population_25yr",
    "norm_Affected_Population_Increase_5to25",
    "norm_Population_Density_per_Hectare",
    "norm_Population_Change_2020_2024_Pct",
    "norm_Elevation_reverse",

    # IDs / labels / geometry / QC-only fields
    "Barangay", "Barangay_No", "psgc_10d", "brgy_code", "city_name",
    "barangay_clean", "barangay_key_strict",
    "Return_Period_Valid_5_25", "flag_25yr_less_than_5yr",
    "geometry",
    "Relative_Population_Density", "Relative Pop Density vs District",
    "Relative Pop Density 2024 vs District",
    "Area_Difference_Pct", "Area Difference Pct",
    "Population_2020",
}

ml_features = [
    c for c in candidate_features
    if c in df_ml.columns and c not in leakage_or_forbidden
]
ml_features = list(dict.fromkeys(ml_features))

missing_manuscript_features = [c for c in candidate_features if c not in df_ml.columns]
if missing_manuscript_features:
    raise ValueError(f"Manuscript ML predictors absent: {missing_manuscript_features}")
if ml_features != candidate_features or len(ml_features) != 11:
    raise ValueError(f"Expected exactly 11 manuscript predictors in defined order, got: {ml_features}")

bad = [c for c in ml_features if c in leakage_or_forbidden]
if bad:
    raise ValueError(f"Leakage/forbidden features selected: {bad}")

if "Elevation_Mean" not in ml_features:
    raise ValueError("Elevation_Mean must be included as a raw ML predictor.")

print("Leakage-free ML features:")
for c in ml_features:
    print(" -", c)
print("Feature count:", len(ml_features))

ml_ready_cols = keep_existing(
    ["Barangay", "Barangay_No", "psgc_10d", "city_name"],
    df_ml
) + ml_features + [target_col]

df_ml_ready = df_ml[ml_ready_cols].copy()
df_ml_ready.to_csv(OUT_ML_READY_CSV, index=False)

print(f"Exported ML-ready dataset: {OUT_ML_READY_CSV} shape={df_ml_ready.shape}")

# Leakage control note:
# Raw Elevation_Mean is allowed as a predictor.
# norm_Elevation_reverse and Vulnerability are DPI-derived/intermediate fields and must not be used as predictors.
for leakage_col in [
    "norm_Elevation_reverse",
    "Vulnerability",
    "DPI",
    "DPI_Scaled",
    "CSI",
    "Exposure",
]:
    assert leakage_col not in ml_features, f"Leakage column included in ML features: {leakage_col}"


Leakage-free ML features:
 - Flood_PCT_5yr
 - Flood_PCT_25yr
 - Flood_Escalation_5to25
 - Flood_Severity_Increase_5to25_Sqm
 - Population_2024
 - Affected_Population_5yr
 - Affected_Population_25yr
 - Affected_Population_Increase_5to25
 - Population_Density_per_Hectare
 - Population_Change_2020_2024_Pct
 - Elevation_Mean
Feature count: 11
Exported ML-ready dataset: outputs/manila_ml_ready_dataset.csv shape=(897, 14)


## 20. Train-Test Split

**Purpose:** Create an 80/20 stratified split by `DPI_Risk_Class` with `random_state=42`.

**Why Both Split and CV are Used:** The held-out test set estimates final generalization once. Cross-validation on the **training set only** supports model selection without touching the test set.


In [21]:
df_model = df_ml[ml_features + [target_col]].dropna(subset=[target_col]).copy()
X = df_model[ml_features]
y = df_model[target_col].astype(str)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Class encoding:")
for name, code_ in zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)):
    print(f"  {name} -> {code_}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.20, random_state=RANDOM_STATE, stratify=y_encoded
)

print("Train size:", X_train.shape, "Test size:", X_test.shape)
print("Train class distribution:", dict(zip(*np.unique(y_train, return_counts=True))))
print("Test class distribution:", dict(zip(*np.unique(y_test, return_counts=True))))


Class encoding:
  High -> 0
  Low -> 1
  Medium -> 2
Train size: (717, 11) Test size: (180, 11)
Train class distribution: {np.int64(0): np.int64(239), np.int64(1): np.int64(239), np.int64(2): np.int64(239)}
Test class distribution: {np.int64(0): np.int64(60), np.int64(1): np.int64(60), np.int64(2): np.int64(60)}


## 21. Model Pipelines

**Purpose:** Define Random Forest, Gradient Boosting, and MLP pipelines with preprocessing inside the pipeline.

- Trees: `SimpleImputer(median) -> Classifier`
- MLP: `SimpleImputer(median) -> StandardScaler -> MLPClassifier`

**Why Pipelines:** Imputation/scaling must fit on training folds only. Fitting on the full dataset before splitting would leak test information.


In [22]:
base_models = {
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced")),
    ]),
    "Gradient Boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", GradientBoostingClassifier(random_state=RANDOM_STATE)),
    ]),
    "MLP": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", MLPClassifier(
            random_state=RANDOM_STATE,
            max_iter=1000,
            early_stopping=True,
        )),
    ]),
}

param_grids = {
    "Random Forest": {
        "model__n_estimators": [200, 300],
        "model__max_depth": [None, 12],
        "model__min_samples_leaf": [1, 2],
    },
    "Gradient Boosting": {
        "model__n_estimators": [100, 200],
        "model__learning_rate": [0.05, 0.1],
        "model__max_depth": [2, 3],
    },
    "MLP": {
        "model__hidden_layer_sizes": [(64, 32), (32,)],
        "model__alpha": [1e-4, 1e-3],
    },
}

print("Model pipelines defined:", list(base_models.keys()))


Model pipelines defined: ['Random Forest', 'Gradient Boosting', 'MLP']


## 22. Cross-Validation and Hyperparameter Tuning

**Purpose:** Stratified 5-fold CV on the **training set** with primary selection metric **F1-macro**, using identical folds across models via a shared `StratifiedKFold`.

**Why F1-Macro:** Treats Low/Medium/High classes equally, which is appropriate for priority-class balance assessment.


In [23]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_rows = []
tuned_models = {}
best_params = {}

for name, pipe in base_models.items():
    print(f"\nTuning {name} with GridSearchCV (training set only)...")
    search = GridSearchCV(
        estimator=pipe,
        param_grid=param_grids[name],
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1,
        refit=True,
    )
    search.fit(X_train, y_train)
    tuned_models[name] = search.best_estimator_
    best_params[name] = search.best_params_

    cv_rows.append({
        "Model": name,
        "CV_F1_Macro_Mean": float(search.best_score_),
        "CV_F1_Macro_Std": float(search.cv_results_["std_test_score"][search.best_index_]),
        "Best_Params": str(search.best_params_),
    })
    print(f"Best CV F1-macro: {search.best_score_:.4f}")
    print("Best params:", search.best_params_)

cv_results_df = pd.DataFrame(cv_rows).sort_values("CV_F1_Macro_Mean", ascending=False)
print("\nCross-validation model selection summary:")
print(cv_results_df.to_string(index=False))



Tuning Random Forest with GridSearchCV (training set only)...
Best CV F1-macro: 0.9312
Best params: {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__n_estimators': 200}

Tuning Gradient Boosting with GridSearchCV (training set only)...
Best CV F1-macro: 0.9343
Best params: {'model__learning_rate': 0.1, 'model__max_depth': 2, 'model__n_estimators': 200}

Tuning MLP with GridSearchCV (training set only)...
Best CV F1-macro: 0.8817
Best params: {'model__alpha': 0.0001, 'model__hidden_layer_sizes': (64, 32)}

Cross-validation model selection summary:
            Model  CV_F1_Macro_Mean  CV_F1_Macro_Std                                                                          Best_Params
Gradient Boosting          0.934322         0.022188     {'model__learning_rate': 0.1, 'model__max_depth': 2, 'model__n_estimators': 200}
    Random Forest          0.931196         0.029691 {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__n_estimators': 200}
              ML

## 23. Final Held-Out Test Evaluation

**Purpose:** Select the best model by training CV F1-macro, ensure it is refit on the full training set (GridSearchCV `refit=True`), then evaluate **once** on the held-out test set.

**Why evaluate the selected model only once on the test set:** Repeated test-set peeking would bias the final generalization estimate. Test metrics are for final reporting, not model selection.

**Reported Metrics** include
- Accuracy
- Balanced Accuracy
- per-class Precision/Recall/F1
- F1-Macro, F1-Weighted
- ROC-AUC OvR Macro (when probabilities exist)
- Confusion Matrix
- Classification Report


In [24]:
def evaluate_model(name, model, X_te, y_te, label_encoder):
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te) if hasattr(model, "predict_proba") else None
    try:
        roc = (
            roc_auc_score(y_te, y_proba, multi_class="ovr", average="macro")
            if y_proba is not None else np.nan
        )
    except Exception:
        roc = np.nan

    precision, recall, f1, support = precision_recall_fscore_support(
        y_te, y_pred, labels=list(range(len(label_encoder.classes_))), zero_division=0
    )
    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_te, y_pred),
        "Balanced_Accuracy": balanced_accuracy_score(y_te, y_pred),
        "F1_Macro": f1_score(y_te, y_pred, average="macro"),
        "F1_Weighted": f1_score(y_te, y_pred, average="weighted"),
        "ROC_AUC_OVR_Macro": roc,
    }
    for i, cls in enumerate(label_encoder.classes_):
        metrics[f"Precision_{cls}"] = precision[i]
        metrics[f"Recall_{cls}"] = recall[i]
        metrics[f"F1_{cls}"] = f1[i]
        metrics[f"Support_{cls}"] = support[i]

    cm = confusion_matrix(y_te, y_pred)
    report = classification_report(y_te, y_pred, target_names=label_encoder.classes_)
    return metrics, cm, report, y_pred


best_model_name = cv_results_df.iloc[0]["Model"]
best_model = tuned_models[best_model_name]
print("Best model by training CV F1-macro:", best_model_name)

test_rows = []
confusion_matrices = {}
classification_reports = {}

for name, model in tuned_models.items():
    print(f"\n==== Held-out test evaluation: {name} ====")
    metrics, cm, report, _ = evaluate_model(name, model, X_test, y_test, label_encoder)
    test_rows.append(metrics)
    confusion_matrices[name] = cm
    classification_reports[name] = report
    print(report)
    print("Confusion matrix:\n", cm)

test_results_df = pd.DataFrame(test_rows)
print("\nSelected best model test metrics:")
print(test_results_df.loc[test_results_df["Model"] == best_model_name].T)


Best model by training CV F1-macro: Gradient Boosting

==== Held-out test evaluation: Random Forest ====
              precision    recall  f1-score   support

        High       0.95      0.92      0.93        60
         Low       0.93      0.92      0.92        60
      Medium       0.84      0.88      0.86        60

    accuracy                           0.91       180
   macro avg       0.91      0.91      0.91       180
weighted avg       0.91      0.91      0.91       180

Confusion matrix:
 [[55  0  5]
 [ 0 55  5]
 [ 3  4 53]]

==== Held-out test evaluation: Gradient Boosting ====
              precision    recall  f1-score   support

        High       0.93      0.93      0.93        60
         Low       0.95      0.93      0.94        60
      Medium       0.87      0.88      0.88        60

    accuracy                           0.92       180
   macro avg       0.92      0.92      0.92       180
weighted avg       0.92      0.92      0.92       180

Confusion matrix:
 [[5

## 24. Model Comparison Table

**Purpose:** Combine training CV selection metrics and held-out test metrics for all three models into one comparison table.

**Output:** `model_comparison_results.csv`


In [25]:
comparison_df = cv_results_df.merge(test_results_df, on="Model", how="left")
comparison_df = comparison_df.sort_values(["CV_F1_Macro_Mean", "F1_Macro"], ascending=False)
comparison_df.to_csv(OUT_MODEL_COMPARISON_CSV, index=False)
print("Model comparison table:")
print(comparison_df.to_string(index=False))
print(f"Saved: {OUT_MODEL_COMPARISON_CSV}")


Model comparison table:
            Model  CV_F1_Macro_Mean  CV_F1_Macro_Std                                                                          Best_Params  Accuracy  Balanced_Accuracy  F1_Macro  F1_Weighted  ROC_AUC_OVR_Macro  Precision_High  Recall_High  F1_High  Support_High  Precision_Low  Recall_Low   F1_Low  Support_Low  Precision_Medium  Recall_Medium  F1_Medium  Support_Medium
Gradient Boosting          0.934322         0.022188     {'model__learning_rate': 0.1, 'model__max_depth': 2, 'model__n_estimators': 200}  0.916667           0.916667  0.916848     0.916848           0.986343        0.933333     0.933333 0.933333            60       0.949153    0.933333 0.941176           60          0.868852       0.883333   0.876033              60
    Random Forest          0.931196         0.029691 {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__n_estimators': 200}  0.905556           0.905556  0.906121     0.906121           0.989769        0.948276     0.91666

## 25. Feature Importance Extraction

**Purpose:** Extract impurity-based feature importance for Random Forest and Gradient Boosting.

**Interpretation:** Values are **model-based importance** for approximating DPI-derived classes. They do **not** imply causal drivers of real-world flooding.

If the selected best model is MLP, tree-model importances are still reported; standard MLP has no direct impurity-based importance.


In [26]:
importance_frames = {}
for name in ["Random Forest", "Gradient Boosting"]:
    model = tuned_models[name]
    clf = model.named_steps["model"]
    if hasattr(clf, "feature_importances_"):
        imp_df = pd.DataFrame({
            "Feature": ml_features,
            "Importance": clf.feature_importances_,
            "Model": name,
        }).sort_values("Importance", ascending=False)
        importance_frames[name] = imp_df
        out_path = OUTPUT_DIR / f"feature_importance_{name.lower().replace(' ', '_')}.csv"
        imp_df.to_csv(out_path, index=False)
        print(f"\nFeature importance ({name}) — model-based only:")
        print(imp_df.to_string(index=False))

if best_model_name == "MLP":
    print(
        "\nNote: Best model is MLP. Standard MLPClassifier does not provide "
        "impurity-based feature importance; tree-model importances above are reported for interpretation support only."
    )



Feature importance (Random Forest) — model-based only:
                           Feature  Importance         Model
          Affected_Population_25yr    0.309513 Random Forest
           Affected_Population_5yr    0.160888 Random Forest
                   Population_2024    0.145088 Random Forest
                    Flood_PCT_25yr    0.138182 Random Forest
                     Flood_PCT_5yr    0.066725 Random Forest
Affected_Population_Increase_5to25    0.060536 Random Forest
 Flood_Severity_Increase_5to25_Sqm    0.030526 Random Forest
            Flood_Escalation_5to25    0.027791 Random Forest
    Population_Density_per_Hectare    0.025165 Random Forest
                    Elevation_Mean    0.023439 Random Forest
   Population_Change_2020_2024_Pct    0.012147 Random Forest

Feature importance (Gradient Boosting) — model-based only:
                           Feature  Importance             Model
          Affected_Population_25yr    0.601811 Gradient Boosting
                    Fl

## 26. Barangay predictions and planning references

**Inputs:** fitted model, 11 predictors, finalized DPI classes. **Sequence:** map DPI classes to illustrative actions; predict classes for all existing barangays; assign a stable display rank; validate 897 complete rows; write the canonical CSV. The displayed **ML confidence is confidence in reproducing an index label**, not the probability of a future flood. Planning references are illustrative, not evacuation instructions.


In [27]:
# 1. Exact mapping from Thesis Table 18
PLANNING_REFERENCE_MAP = {
    "Low": "Routine monitoring, drainage inspection, annual information updating, and regular drills.",
    "Medium": "Periodic drainage/waterway clearing, more frequent early-warning checks, community drills, relief-stock review, and evacuation-route verification.",
    "High": "Priority site validation, detailed evacuation planning, pre-positioning review, infrastructure assessment, and enhanced preparedness review."
}

DRRM_PILLAR_MAP = {
    "Low": "Disaster Prevention and Mitigation; Disaster Preparedness",
    "Medium": "Disaster Prevention and Mitigation; Disaster Preparedness",
    "High": "Disaster Prevention and Mitigation; Disaster Preparedness; Disaster Response and Early Recovery"
}

# 2. Apply the mapping to the final prediction dataframe
df_ml["Planning_Reference"] = df_ml["DPI_Risk_Class"].map(PLANNING_REFERENCE_MAP)
df_ml["DRRM_Pillar"] = df_ml["DPI_Risk_Class"].map(DRRM_PILLAR_MAP)

# 3. Safety check — must run BEFORE exporting, not after
unmapped = df_ml[df_ml[["Planning_Reference", "DRRM_Pillar"]].isna().any(axis=1)]
if len(unmapped) > 0:
    raise ValueError(
        f"⚠️ {len(unmapped)} barangay(s) have an unrecognized DPI_Risk_Class value "
        f"that didn't match Low/Medium/High: {unmapped['DPI_Risk_Class'].unique().tolist()}"
    )
else:
    print(f"✅ All {len(df_ml)} barangays successfully mapped to Planning_Reference and DRRM_Pillar.")

# The results CSV is exported after predictions, validation, and ranking in the following cell.


✅ All 897 barangays successfully mapped to Planning_Reference and DRRM_Pillar.


In [28]:
X_all = df_ml[ml_features].copy()
df_ml["ML_Predicted_Class_Encoded"] = best_model.predict(X_all)
df_ml["ML_Predicted_Risk_Class"] = label_encoder.inverse_transform(df_ml["ML_Predicted_Class_Encoded"])
if not hasattr(best_model, "predict_proba"):
    raise ValueError("The selected classifier must provide prediction probabilities for the results CSV.")
df_ml["ML_Prediction_Confidence"] = best_model.predict_proba(X_all).max(axis=1)
df_ml["Best_Model"] = best_model_name

# Barangay_No may repeat (e.g., names with suffixes); use the normalized name
# as a deterministic tie-breaker, without changing DPI values or classes.
rank_key = df_ml["barangay_key_strict"].astype("string").str.strip()
if rank_key.isna().any() or rank_key.eq("").any() or rank_key.duplicated().any():
    raise ValueError("Nonblank unique barangay names are needed for deterministic ranks; inspect source names.")
df_ml = (df_ml.assign(_rank_key=rank_key)
         .sort_values(["DPI_Scaled", "_rank_key"], ascending=[False, True], kind="stable")
         .drop(columns="_rank_key").reset_index(drop=True))
df_ml["DPI_Rank"] = np.arange(1, len(df_ml) + 1)

pred_cols = keep_existing(
    [
        "Barangay", "Barangay_No", "psgc_10d", "city_name",
        "Population_2020", "Population_2024",
        "Population_Density_per_Hectare",
        "Population_Change_2020_2024_Pct",
        "Elevation_Mean", "Elevation_Median", "Elevation_Min", "Elevation_Max",
        "Flood_PCT_5yr", "Flood_PCT_25yr",
        "Flood_Escalation_5to25", "Flood_Severity_Increase_5to25_Sqm",
        "Affected_Population_5yr", "Affected_Population_25yr",
        "Affected_Population_Increase_5to25",
        "Hazard", "Exposure", "Vulnerability",
        "DPI", "DPI_Scaled", "DPI_Risk_Class", "DPI_Rank",
        "ML_Predicted_Risk_Class", "ML_Prediction_Confidence", "Best_Model",
        "Vulnerability_Elevation_Included", "Planning_Reference", "DRRM_Pillar"
    ],
    df_ml,
)
required_results = [
    "Barangay", "Population_2024", "Flood_PCT_5yr", "Flood_PCT_25yr",
    "Flood_Escalation_5to25", "Flood_Severity_Increase_5to25_Sqm",
    "Affected_Population_5yr", "Affected_Population_25yr",
    "Affected_Population_Increase_5to25", "Population_Density_per_Hectare",
    "Population_Change_2020_2024_Pct", "Elevation_Mean",
    "Hazard", "Exposure", "Vulnerability", "DPI", "DPI_Scaled",
    "DPI_Risk_Class", "DPI_Rank", "ML_Predicted_Risk_Class",
    "ML_Prediction_Confidence", "Planning_Reference", "DRRM_Pillar",
]
missing = [c for c in required_results if c not in df_ml.columns]
if missing:
    raise ValueError(f"Results CSV missing fields: {missing}")
predictions_out = df_ml[pred_cols].copy()
assert_exact_barangay_count(predictions_out, "results_csv")
nulls = predictions_out[required_results].isna().sum()
if nulls.any():
    raise ValueError(f"Required results contain nulls: {nulls[nulls > 0].to_dict()}")
if not predictions_out["DPI_Scaled"].between(0, 100).all():
    raise ValueError("DPI_Scaled outside 0..100")
if not predictions_out["ML_Prediction_Confidence"].between(0, 1).all():
    raise ValueError("Invalid ML prediction confidence")
if predictions_out["DPI_Rank"].tolist() != list(range(1, EXPECTED_BARANGAY_COUNT + 1)):
    raise ValueError("Invalid rank sequence")
counts = predictions_out["DPI_Risk_Class"].astype(str).value_counts()
if set(counts.index) != {"Low", "Medium", "High"} or not counts.eq(299).all():
    raise ValueError(f"Expected three balanced tertile groups; got {counts.to_dict()}")
if set(predictions_out["ML_Predicted_Risk_Class"].astype(str)) - {"Low", "Medium", "High"}:
    raise ValueError("Unexpected model class label")
for klass in ("Low", "Medium", "High"):
    rows = predictions_out["DPI_Risk_Class"].astype(str).eq(klass)
    if not predictions_out.loc[rows, "Planning_Reference"].eq(PLANNING_REFERENCE_MAP[klass]).all():
        raise ValueError(f"Planning reference mismatch for {klass}")
    if not predictions_out.loc[rows, "DRRM_Pillar"].eq(DRRM_PILLAR_MAP[klass]).all():
        raise ValueError(f"DRRM pillar mismatch for {klass}")

predictions_out.to_csv(OUT_PREDICTIONS_CSV, index=False)
print(f"Saved {len(predictions_out)} barangay results to {OUT_PREDICTIONS_CSV}")
display(predictions_out.head(10))


[results_csv] Barangay count validated: 897
Saved 897 barangay results to outputs/barangay_flood_risk_predictions.csv


,Barangay,Barangay_No,Population_2020,Population_2024,Population_Density_per_Hectare,Population_Change_2020_2024_Pct,Elevation_Mean,Elevation_Median,Elevation_Min,Elevation_Max,Flood_PCT_5yr,Flood_PCT_25yr,Flood_Escalation_5to25,Flood_Severity_Increase_5to25_Sqm,Affected_Population_5yr,Affected_Population_25yr,Affected_Population_Increase_5to25,Hazard,Exposure,Vulnerability,DPI,DPI_Scaled,DPI_Risk_Class,DPI_Rank,ML_Predicted_Risk_Class,ML_Prediction_Confidence,Best_Model,Vulnerability_Elevation_Included,Planning_Reference,DRRM_Pillar
0,Barangay 310,310,13602,13761.0,1590.486639,1.168946,2.000000,2.0,2,2,43.280626,98.959786,55.679161,48173.993550,5955.846894,13617.856203,7662.009309,0.613507,0.847808,0.427736,0.778408,100.000000,High,1,High,0.999874,Gradient Boosting,True,"Priority site validation, detailed evacuation ...",Disaster Prevention and Mitigation; Disaster P...
1,Barangay 628,628,15202,16279.0,884.899544,7.084594,1.408893,2.0,0,3,49.365086,85.649947,36.284861,66751.220510,8036.142361,13942.954817,5906.812457,0.568351,0.845205,0.392955,0.765744,98.359864,High,2,High,0.999792,Gradient Boosting,True,"Priority site validation, detailed evacuation ...",Disaster Prevention and Mitigation; Disaster P...
2,Barangay 649,649,64750,69769.0,632.795155,7.751351,0.625217,1.0,0,6,10.831892,17.449150,6.617259,72958.764246,7557.302478,12174.097732,4616.795254,0.220661,0.731607,0.400730,0.613995,78.706502,High,3,High,0.994593,Gradient Boosting,True,"Priority site validation, detailed evacuation ...",Disaster Prevention and Mitigation; Disaster P...
3,Barangay 598,598,12027,10510.0,1368.152765,-12.613287,2.567198,3.0,1,4,99.990275,100.000000,0.009725,7.470510,10508.977920,10510.000000,1.022080,0.500019,0.584640,0.382022,0.556641,71.278457,High,4,High,0.998127,Gradient Boosting,True,"Priority site validation, detailed evacuation ...",Disaster Prevention and Mitigation; Disaster P...
4,Barangay 648,648,5634,9086.0,2149.392097,61.270856,1.249852,1.0,0,2,62.576214,97.397830,34.821616,14719.938939,5685.674828,8849.566869,3163.892042,0.533887,0.529554,0.529859,0.530407,67.880792,High,5,High,0.999894,Gradient Boosting,True,"Priority site validation, detailed evacuation ...",Disaster Prevention and Mitigation; Disaster P...
5,Barangay 20,20,45772,43691.0,318.321697,-4.546448,1.223006,1.0,0,3,17.329451,23.628010,6.298559,86450.381573,7571.410565,10323.313783,2751.903217,0.275306,0.606676,0.346875,0.527794,67.542399,High,6,High,0.997558,Gradient Boosting,True,"Priority site validation, detailed evacuation ...",Disaster Prevention and Mitigation; Disaster P...
6,Barangay 843,843,8744,9355.0,3282.855633,6.987649,1.479754,1.0,1,2,100.000000,100.000000,0.000000,0.000000,9355.000000,9355.000000,0.000000,0.500000,0.520380,0.591139,0.520528,66.601326,High,7,High,0.997593,Gradient Boosting,True,"Priority site validation, detailed evacuation ...",Disaster Prevention and Mitigation; Disaster P...
7,Barangay 900,900,8581,8689.0,1335.142407,1.258595,1.952344,2.0,1,3,51.380978,96.846452,45.465474,29588.567001,4464.493170,8414.988226,3950.495055,0.563866,0.514651,0.408132,0.518003,66.274272,High,8,High,0.999895,Gradient Boosting,True,"Priority site validation, detailed evacuation ...",Disaster Prevention and Mitigation; Disaster P...
8,Barangay 397,397,4175,8835.0,2143.964363,111.616766,2.000000,2.0,2,2,89.079978,99.757333,10.677355,4400.000000,7870.216037,8813.560356,943.344320,0.512965,0.501380,0.525475,0.505003,64.590671,High,9,High,0.999790,Gradient Boosting,True,"Priority site validation, detailed evacuation ...",Disaster Prevention and Mitigation; Disaster P...
9,Barangay 898,898,7596,7812.0,1306.755912,2.843602,1.844770,2.0,1,3,71.408698,91.803502,20.394805,12192.346987,5578.447453,7171.689603,1593.242150,0.492822,0.417709,0.410415,0.431775,55.106718,High,10,High,0.999751,Gradient Boosting,True,"Priority site validation, detailed evacuation ...",Disaster Prevention and Mitigation; Disaster P...


## 27. Serialized model and output checkpoint

Save the fitted sklearn pipeline, its feature order, and label encoder as a **server-side** joblib bundle. Verify the standard thesis outputs. **Never deserialize untrusted joblib files.**


In [29]:
joblib.dump(
    {
        "pipeline": best_model,
        "label_encoder": label_encoder,
        "features": ml_features,
        "best_model_name": best_model_name,
        "best_params": best_params.get(best_model_name),
        "random_state": RANDOM_STATE,
    },
    OUT_BEST_MODEL_JOBLIB,
)

required_outputs = [
    OUT_FLOOD_CSV,
    OUT_ENRICHED_CSV,
    OUT_ENRICHED_XLSX,
    OUT_ML_READY_CSV,
    OUT_MODEL_COMPARISON_CSV,
    OUT_PREDICTIONS_CSV,
    OUT_BEST_MODEL_JOBLIB,
]

print("\nOutput file checklist:")
for path in required_outputs:
    status = "CHECK" if path.exists() else "MISSING"
    print(f"  [{status}] {path}")

missing_outputs = [p for p in required_outputs if not p.exists()]
if missing_outputs:
    critical = [p for p in missing_outputs if p.suffix.lower() != ".xlsx"]
    if critical:
        raise FileNotFoundError(f"Missing required outputs: {critical}")
    print("WARNING: Excel output missing (non-critical if openpyxl unavailable).")

print("\nBest model:", best_model_name)
print("Feature count:", len(ml_features))
print("Barangay count:", len(df_ml))

print("Elevation CSV integrated:", "Elevation_Mean" in df_ml.columns and df_ml["Elevation_Mean"].notna().any())
if "Elevation_Mean" in df_ml.columns:
    print("Missing elevation values:", int(df_ml["Elevation_Mean"].isna().sum()))
    if REQUIRE_ELEVATION:
        assert df_ml["Elevation_Mean"].isna().sum() == 0, "Missing required elevation values."
assert df_ml["DPI_Risk_Class"].nunique() == 3, "DPI risk classification did not produce three classes."



Output file checklist:
  [CHECK] outputs/MetroManila_Barangay_Flood_Area_Validated.csv
  [CHECK] outputs/manila-city-flood-population_enriched_NEW.csv
  [CHECK] outputs/manila-city-flood-population_enriched_NEW.xlsx
  [CHECK] outputs/manila_ml_ready_dataset.csv
  [CHECK] outputs/model_comparison_results.csv
  [CHECK] outputs/barangay_flood_risk_predictions.csv
  [CHECK] outputs/best_flood_risk_model_pipeline.joblib

Best model: Gradient Boosting
Feature count: 11
Barangay count: 897
Elevation CSV integrated: True
Missing elevation values: 0


## Defense snapshot | Evidence from the **current run**

This short panel-facing checkpoint displays only values computed **by the run**, not hard-coded example scores. Read it after the results CSV export. The charts/tables and detailed audits remain below; the PDF report is in the appendix at the end.

- **Method:** Equal indicator averages → three component scores → entropy weighting → tertile DPI classes.
- **Selection:** Choose the model by training-fold CV F1-macro; describe held-out test metrics only as agreement with DPI labels.
- **Prototype:** The predictions CSV is the single barangay-level results file for the prototype; map geometry is not produced here.


In [30]:
# Read-only defense snapshot based on the canonical results CSV.
from IPython.display import display, Markdown

if not OUT_PREDICTIONS_CSV.exists():
    raise FileNotFoundError("Run the predictions CSV export before presenting this snapshot.")
snapshot_df = pd.read_csv(OUT_PREDICTIONS_CSV)
if len(snapshot_df) != EXPECTED_BARANGAY_COUNT:
    raise ValueError("The results CSV does not contain all 897 barangays.")
display(Markdown("### 1 · Run integrity and results CSV"))
display(pd.DataFrame([{
    "Barangays": len(snapshot_df),
    "ML predictors": len(ml_features),
    "Selected model": best_model_name,
    "Selection rule": "Training CV F1-macro",
    "Prototype input": OUT_PREDICTIONS_CSV.name,
}]))
display(Markdown("### 2 · Entropy weights (components, not individual indicators)"))
display(pd.DataFrame([{"Component": c, "Weight": float(entropy_w[c])}
                      for c in ("Hazard", "Exposure", "Vulnerability")]).round(6))
display(Markdown("### 3 · DPI priority class distribution"))
display(pd.DataFrame([{"Priority": c,
                       "Barangays": int(snapshot_df["DPI_Risk_Class"].astype(str).eq(c).sum())}
                      for c in ("Low", "Medium", "High")]))
display(Markdown("### 4 · RF / GB / MLP comparison (use training CV for selection)"))
display(comparison_df.reset_index(drop=True).round(4))
display(Markdown("### 5 · Illustrative high-priority records — not flood forecasts"))
display(snapshot_df[["Barangay", "DPI_Rank", "DPI_Scaled", "DPI_Risk_Class",
                     "ML_Predicted_Risk_Class", "Planning_Reference"]].head(5).round(3))
print("Source: results CSV from this run. Output directory:", OUTPUT_DIR.resolve())


### 1 · Run integrity and results CSV

,Barangays,ML predictors,Selected model,Selection rule,Prototype input
0,897,11,Gradient Boosting,Training CV F1-macro,barangay_flood_risk_predictions.csv


### 2 · Entropy weights (components, not individual indicators)

,Component,Weight
0,Hazard,0.192859
1,Exposure,0.749503
2,Vulnerability,0.057638


### 3 · DPI priority class distribution

...

### 4 · RF / GB / MLP comparison (use training CV for selection)

,Model,CV_F1_Macro_Mean,CV_F1_Macro_Std,Best_Params,Accuracy,Balanced_Accuracy,F1_Macro,F1_Weighted,ROC_AUC_OVR_Macro,Precision_High,Recall_High,F1_High,Support_High,Precision_Low,Recall_Low,F1_Low,Support_Low,Precision_Medium,Recall_Medium,F1_Medium,Support_Medium
0,Gradient Boosting,0.9343,0.0222,"{'model__learning_rate': 0.1, 'model__max_dept...",0.9167,0.9167,0.9168,0.9168,0.9863,0.9333,0.9333,0.9333,60,0.9492,0.9333,0.9412,60,0.8689,0.8833,0.8760,60
1,Random Forest,0.9312,0.0297,"{'model__max_depth': None, 'model__min_samples...",0.9056,0.9056,0.9061,0.9061,0.9898,0.9483,0.9167,0.9322,60,0.9322,0.9167,0.9244,60,0.8413,0.8833,0.8618,60
2,MLP,0.8817,0.0401,"{'model__alpha': 0.0001, 'model__hidden_layer_...",0.9333,0.9333,0.9336,0.9336,0.9910,0.9500,0.9500,0.9500,60,0.9655,0.9333,0.9492,60,0.8871,0.9167,0.9016,60


### 5 · Illustrative high-priority records — not flood forecasts

,Barangay,DPI_Rank,DPI_Scaled,DPI_Risk_Class,ML_Predicted_Risk_Class,Planning_Reference
0,Barangay 310,1,100.000,High,High,"Priority site validation, detailed evacuation ..."
1,Barangay 628,2,98.360,High,High,"Priority site validation, detailed evacuation ..."
2,Barangay 649,3,78.707,High,High,"Priority site validation, detailed evacuation ..."
3,Barangay 598,4,71.278,High,High,"Priority site validation, detailed evacuation ..."
4,Barangay 648,5,67.881,High,High,"Priority site validation, detailed evacuation ..."


Source: results CSV from this run. Output directory: /content/outputs


## 28. Final Reproducibility and Limitation Notes

**Reproducibility**
- Fixed `random_state=42` for split, CV folds, and model estimators.
- Preprocessing is confined to sklearn pipelines / GridSearchCV folds.
- Model selection uses training CV F1-macro only; held-out test is evaluated after selection.
- Intermediate and final outputs are written to the configured `OUTPUT_DIR`.
- Elevation is supplied through the required QGIS Zonal Statistics output file.

**Methodological Limitations**
- LiPAD flood layers are modeled hazard surfaces, not observed event inundation maps.
- The 100-year layer is excluded from DPI/ML due to return-period consistency concerns.
- DPI risk classes are index-derived proxy labels, not ground-truth flood impacts.
- Supervised metrics measure agreement with the deterministic DPI classification, not real-world forecast skill.
- Elevation is represented as barangay-level summary elevation and does not capture micro-scale drainage, road depressions, drainage capacity, land subsidence, or building-level exposure.
- Population Change 2020–2024 (%) is a four-year observed change, not an annualized growth rate.
- District-context features, if computed, are not among the 11 manuscript ML predictors.

**Use Statement**
- Prototype outputs support barangay-level planning prioritization research for the City of Manila. They must not be interpreted as official flood warnings, evacuation orders, or real-time flood predictions.

In [31]:
print("=" * 72)
print("Manila flood risk complete pipeline finished.")
print(f"Barangays: {len(df_ml)} (expected {EXPECTED_BARANGAY_COUNT})")
print(f"Best model: {best_model_name}")
print(f"Outputs directory: {OUTPUT_DIR}")
print("=" * 72)


Manila flood risk complete pipeline finished.
Barangays: 897 (expected 897)
Best model: Gradient Boosting
Outputs directory: outputs


### QGIS Elevation Data Update

The vulnerability component uses barangay-level mean elevation extracted from a DEM/DTM raster using QGIS Zonal Statistics. The Manila barangay polygon layer served as the zone layer, while the DEM/DTM raster served as the elevation raster.

The QGIS output fields are standardized as `Elevation_Count`, `Elevation_Mean`, `Elevation_Median`, `Elevation_Min`, and `Elevation_Max`.

For the DPI, `Elevation_Mean` is used as the physical susceptibility indicator. It is reverse-normalized so that lower terrain elevation contributes to higher vulnerability. Raw `Elevation_Mean` is permitted as an ML predictor, while `norm_Elevation_reverse` and `Vulnerability` are excluded from the ML feature set to prevent data leakage.


# Additional Thesis Diagnostics and Robustness Checks

The following cells are added to support interpretation of the final DPI and ML results. They should be run after the main pipeline has created `df_ml`, `ml_features`, `best_model`, `best_model_name`, `X_test`, `y_test`, and `OUTPUT_DIR`.


## 100-Year Flood Layer Exclusion Note

The 100-year flood fields may remain in intermediate enriched files for traceability only. They are excluded from Hazard Score, DPI construction, and ML feature selection because the layer failed the expected non-decreasing return-period consistency check in many barangays. The retained flood-hazard inputs for the main analysis are the validated 5-year and 25-year LiPAD layers.


In [32]:
# Helper: retain only columns that exist in the current dataframe
try:
    keep_existing
except NameError:
    def keep_existing(cols, df):
        return [c for c in cols if c in df.columns]

# Robust aliases in case the notebook uses CSI instead of Hazard
hazard_col = "Hazard" if "Hazard" in df_ml.columns else ("CSI" if "CSI" in df_ml.columns else None)
print("Hazard/CSI column used:", hazard_col)

Hazard/CSI column used: Hazard


## A. DPI Component Diagnostics by Risk Class

This section summarizes component behavior by risk class. It is useful for explaining whether the final DPI is hazard-dominant, exposure-dominant, or vulnerability-dominant.


In [33]:
agg_dict = {
    "Barangay_Count": ("Barangay", "count"),
    "Mean_DPI": ("DPI_Scaled", "mean"),
    "Min_DPI": ("DPI_Scaled", "min"),
    "Max_DPI": ("DPI_Scaled", "max"),
    "Mean_Exposure": ("Exposure", "mean"),
    "Mean_Vulnerability": ("Vulnerability", "mean"),
    "Mean_Flood_25yr": ("Flood_PCT_25yr", "mean"),
    "Mean_Population_2024": ("Population_2024", "mean"),
    "Mean_Elevation": ("Elevation_Mean", "mean"),
}
if hazard_col:
    agg_dict["Mean_Hazard_or_CSI"] = (hazard_col, "mean")

component_class_summary = df_ml.groupby("DPI_Risk_Class").agg(**agg_dict).reset_index()
component_class_summary.to_csv(OUTPUT_DIR / "dpi_component_class_summary.csv", index=False)
display(component_class_summary)

corr_cols = keep_existing([hazard_col, "Exposure", "Vulnerability", "DPI_Scaled", "Flood_PCT_25yr", "Population_2024", "Elevation_Mean"], df_ml)
component_corr = df_ml[corr_cols].corr()
display(component_corr.round(3))

,DPI_Risk_Class,Barangay_Count,Mean_DPI,Min_DPI,Max_DPI,Mean_Exposure,Mean_Vulnerability,Mean_Flood_25yr,Mean_Population_2024,Mean_Elevation,Mean_Hazard_or_CSI
0,Low,299,11.684706,0.000000,17.684162,0.033622,0.314336,51.225397,1584.073579,2.872936,0.275764
1,Medium,299,20.209992,17.694624,22.888783,0.064449,0.335138,94.323956,1234.913043,2.127793,0.491063
2,High,299,30.235759,22.929698,100.000000,0.163677,0.374992,92.701533,3544.190635,1.983937,0.494916


,Hazard,Exposure,Vulnerability,DPI_Scaled,Flood_PCT_25yr,Population_2024,Elevation_Mean
Hazard,1.000,0.272,0.178,0.599,0.983,-0.139,-0.420
Exposure,0.272,1.000,0.381,0.932,0.217,0.693,-0.184
Vulnerability,0.178,0.381,1.000,0.424,0.189,0.188,-0.685
DPI_Scaled,0.599,0.932,0.424,1.000,0.549,0.521,-0.338
Flood_PCT_25yr,0.983,0.217,0.189,0.549,1.000,-0.196,-0.410
Population_2024,-0.139,0.693,0.188,0.521,-0.196,1.000,-0.050
Elevation_Mean,-0.420,-0.184,-0.685,-0.338,-0.410,-0.050,1.000


## B. Top-Rank Explanation Table

This table helps explain why some barangays rank highly despite medium flood percentages. The DPI is a disaster-prioritization index, so high population exposure can raise the ranking even when flood coverage percentage is not among the highest.


In [34]:
top_rank_cols = keep_existing([
    "Barangay", "Barangay_No", "Population_2024",
    "Flood_PCT_5yr", "Flood_PCT_25yr",
    "Affected_Population_5yr", "Affected_Population_25yr", "Affected_Population_Increase_5to25",
    "Population_Density_per_Hectare", "Population_Change_2020_2024_Pct",
    "Elevation_Mean", "Elevation_Median", "Elevation_Min", "Elevation_Max",
    hazard_col, "Exposure", "Vulnerability", "DPI_Scaled", "DPI_Risk_Class"
], df_ml)

top_rank_explanation = df_ml[top_rank_cols].sort_values("DPI_Scaled", ascending=False).head(25)
top_rank_explanation.to_csv(OUTPUT_DIR / "top_rank_explanation.csv", index=False)
display(top_rank_explanation)

,Barangay,Barangay_No,Population_2024,Flood_PCT_5yr,Flood_PCT_25yr,Affected_Population_5yr,Affected_Population_25yr,Affected_Population_Increase_5to25,Population_Density_per_Hectare,Population_Change_2020_2024_Pct,Elevation_Mean,Elevation_Median,Elevation_Min,Elevation_Max,Hazard,Exposure,Vulnerability,DPI_Scaled,DPI_Risk_Class
0,Barangay 310,310,13761.0,43.280626,98.959786,5955.846894,13617.856203,7662.009309,1590.486639,1.168946,2.000000,2.0,2,2,0.613507,0.847808,0.427736,100.000000,High
1,Barangay 628,628,16279.0,49.365086,85.649947,8036.142361,13942.954817,5906.812457,884.899544,7.084594,1.408893,2.0,0,3,0.568351,0.845205,0.392955,98.359864,High
2,Barangay 649,649,69769.0,10.831892,17.449150,7557.302478,12174.097732,4616.795254,632.795155,7.751351,0.625217,1.0,0,6,0.220661,0.731607,0.400730,78.706502,High
3,Barangay 598,598,10510.0,99.990275,100.000000,10508.977920,10510.000000,1.022080,1368.152765,-12.613287,2.567198,3.0,1,4,0.500019,0.584640,0.382022,71.278457,High
4,Barangay 648,648,9086.0,62.576214,97.397830,5685.674828,8849.566869,3163.892042,2149.392097,61.270856,1.249852,1.0,0,2,0.533887,0.529554,0.529859,67.880792,High
5,Barangay 20,20,43691.0,17.329451,23.628010,7571.410565,10323.313783,2751.903217,318.321697,-4.546448,1.223006,1.0,0,3,0.275306,0.606676,0.346875,67.542399,High
6,Barangay 843,843,9355.0,100.000000,100.000000,9355.000000,9355.000000,0.000000,3282.855633,6.987649,1.479754,1.0,1,2,0.500000,0.520380,0.591139,66.601326,High
7,Barangay 900,900,8689.0,51.380978,96.846452,4464.493170,8414.988226,3950.495055,1335.142407,1.258595,1.952344,2.0,1,3,0.563866,0.514651,0.408132,66.274272,High
8,Barangay 397,397,8835.0,89.079978,99.757333,7870.216037,8813.560356,943.344320,2143.964363,111.616766,2.000000,2.0,2,2,0.512965,0.501380,0.525475,64.590671,High
9,Barangay 898,898,7812.0,71.408698,91.803502,5578.447453,7171.689603,1593.242150,1306.755912,2.843602,1.844770,2.0,1,3,0.492822,0.417709,0.410415,55.106718,High


## C. High-DPI but Medium-Flood Audit

This audit identifies High-priority barangays whose 25-year flood percentage is below the citywide median. These are not automatic errors; they usually indicate that population exposure or vulnerability raised the final priority score.


In [35]:
median_flood_25 = df_ml["Flood_PCT_25yr"].median()
high_dpi_medium_flood = df_ml[
    (df_ml["DPI_Risk_Class"] == "High") &
    (df_ml["Flood_PCT_25yr"] < median_flood_25)
][top_rank_cols].sort_values("DPI_Scaled", ascending=False)

high_dpi_medium_flood.to_csv(OUTPUT_DIR / "high_dpi_medium_flood_audit.csv", index=False)
print(f"Median Flood_PCT_25yr: {median_flood_25:.4f}")
print(f"High-priority barangays below median 25-year flood percentage: {len(high_dpi_medium_flood)}")
display(high_dpi_medium_flood.head(25))

Median Flood_PCT_25yr: 96.4480
High-priority barangays below median 25-year flood percentage: 106


,Barangay,Barangay_No,Population_2024,Flood_PCT_5yr,Flood_PCT_25yr,Affected_Population_5yr,Affected_Population_25yr,Affected_Population_Increase_5to25,Population_Density_per_Hectare,Population_Change_2020_2024_Pct,Elevation_Mean,Elevation_Median,Elevation_Min,Elevation_Max,Hazard,Exposure,Vulnerability,DPI_Scaled,DPI_Risk_Class
1,Barangay 628,628,16279.0,49.365086,85.649947,8036.142361,13942.954817,5906.812457,884.899544,7.084594,1.408893,2.0,0,3,0.568351,0.845205,0.392955,98.359864,High
2,Barangay 649,649,69769.0,10.831892,17.449150,7557.302478,12174.097732,4616.795254,632.795155,7.751351,0.625217,1.0,0,6,0.220661,0.731607,0.400730,78.706502,High
5,Barangay 20,20,43691.0,17.329451,23.628010,7571.410565,10323.313783,2751.903217,318.321697,-4.546448,1.223006,1.0,0,3,0.275306,0.606676,0.346875,67.542399,High
9,Barangay 898,898,7812.0,71.408698,91.803502,5578.447453,7171.689603,1593.242150,1306.755912,2.843602,1.844770,2.0,1,3,0.492822,0.417709,0.410415,55.106718,High
10,Barangay 101,101,13659.0,36.082877,55.173508,4928.560169,7536.149491,2607.589322,1313.398604,-11.062638,1.901876,2.0,0,4,0.322489,0.449937,0.402419,53.920912,High
13,Barangay 830,830,5407.0,77.272197,94.990775,4178.107680,5136.151196,958.043516,128.975621,4.161048,1.164181,1.0,0,3,0.617339,0.296994,0.337215,45.952657,High
14,Barangay 128,128,22298.0,23.812298,30.910285,5309.666308,6892.375305,1582.708997,209.459546,-5.923551,9.141101,5.0,0,30,0.292872,0.402048,0.048382,45.889679,High
15,Barangay 183,183,8512.0,63.467876,69.795827,5402.385578,5941.020816,538.635238,2067.236623,1.044634,2.061931,2.0,0,3,0.357379,0.336823,0.465345,44.282086,High
16,Barangay 630,630,6552.0,54.691102,76.565110,3583.361027,5016.546001,1433.184974,256.764781,40.600858,2.352722,3.0,0,7,0.494926,0.295941,0.321530,42.675743,High
17,Barangay 275,275,20995.0,19.816408,29.280518,4160.454863,6147.444755,1986.989892,778.538836,5.987178,1.365465,1.0,0,6,0.197339,0.365375,0.385121,42.457407,High


## D. Elevation Outlier Audit and Note

Elevation is reverse-normalized in the vulnerability component, but a high-population barangay can still rank highly if its exposure score is high. Elevation outliers should be retained unless confirmed as data errors, and should be discussed as diagnostic limitations.


In [36]:
elevation_cols = keep_existing([
    "Barangay", "Barangay_No", "Elevation_Mean", "Elevation_Median", "Elevation_Min", "Elevation_Max", "Elevation_Count",
    "Flood_PCT_25yr", "Population_2024", "Affected_Population_25yr",
    "Vulnerability", "Exposure", "DPI_Scaled", "DPI_Risk_Class"
], df_ml)

elevation_outlier_audit = df_ml[elevation_cols].sort_values("Elevation_Mean", ascending=False)
elevation_outlier_audit.to_csv(OUTPUT_DIR / "elevation_outlier_audit.csv", index=False)
display(elevation_outlier_audit.head(15))
print(f"Zero mean-elevation barangays: {(df_ml['Elevation_Mean'] == 0).sum()}")

,Barangay,Barangay_No,Elevation_Mean,Elevation_Median,Elevation_Min,Elevation_Max,Elevation_Count,Flood_PCT_25yr,Population_2024,Affected_Population_25yr,Vulnerability,Exposure,DPI_Scaled,DPI_Risk_Class
14,Barangay 128,128,9.141101,5.0,0,30,11552,30.910285,22298.0,6892.375305,0.048382,4.020477e-01,45.889679,High
697,Barangay 372,372,8.761012,9.0,2,14,21771,37.181495,4332.0,1610.702357,0.056325,9.394830e-02,15.554043,Low
895,Barangay 191,191,8.657143,9.0,8,9,420,2.918486,754.0,22.005388,0.114869,1.342164e-03,0.564836,Low
896,Barangay 188,188,8.517986,9.0,7,9,556,0.000000,872.0,0.000000,0.108962,5.524718e-13,0.000000,Low
874,Barangay 189,189,8.205579,8.0,7,9,968,10.845029,1618.0,175.472577,0.127933,1.031926e-02,2.572811,Low
852,Barangay 192,192,7.760294,8.0,5,9,1360,22.525978,1771.0,398.935070,0.124817,2.304529e-02,5.291384,Low
859,Barangay 585,585,7.743298,8.0,5,10,2723,22.262000,1138.0,253.341557,0.100593,1.480714e-02,4.395607,Low
894,Barangay 190,190,7.433387,8.0,5,9,623,0.884811,1282.0,11.343279,0.166722,6.360014e-04,0.604132,Low
886,Barangay 587,587,7.288049,7.0,4,13,2937,4.299139,2685.0,115.431889,0.128076,7.018526e-03,1.450415,Low
873,Barangay 581,581,6.801135,7.0,2,10,7226,8.912027,2212.0,197.134027,0.124472,1.170986e-02,2.636512,Low


Zero mean-elevation barangays: 4


## E. Population Change Sensitivity Check

This robustness check tests whether extreme percentage changes from small baseline populations affect the Vulnerability component. The main DPI is unchanged unless the researcher explicitly decides to adopt the winsorized version.


In [37]:
pop_change_col = "Population_Change_2020_2024_Pct"
low_q = df_ml[pop_change_col].quantile(0.01)
high_q = df_ml[pop_change_col].quantile(0.99)

df_ml["Population_Change_2020_2024_Pct_Winsorized"] = df_ml[pop_change_col].clip(lower=low_q, upper=high_q)
df_ml["norm_Population_Change_2020_2024_Pct_Winsorized"] = minmax_normalize(df_ml["Population_Change_2020_2024_Pct_Winsorized"])

vuln_winsor_cols = keep_existing([
    "norm_Population_Density_per_Hectare",
    "norm_Population_Change_2020_2024_Pct_Winsorized",
    "norm_Elevation_reverse",
], df_ml)

df_ml["Vulnerability_WinsorizedPopChange"] = df_ml[vuln_winsor_cols].mean(axis=1)
sensitivity_corr = df_ml[["Vulnerability", "Vulnerability_WinsorizedPopChange"]].corr().iloc[0, 1]
df_ml["Vulnerability_Sensitivity_Diff"] = df_ml["Vulnerability_WinsorizedPopChange"] - df_ml["Vulnerability"]

population_change_sensitivity_audit = df_ml[keep_existing([
    "Barangay", "Population_2020", "Population_2024", pop_change_col,
    "Population_Change_2020_2024_Pct_Winsorized",
    "Vulnerability", "Vulnerability_WinsorizedPopChange", "Vulnerability_Sensitivity_Diff",
    "DPI_Scaled", "DPI_Risk_Class"
], df_ml)].sort_values("Vulnerability_Sensitivity_Diff", key=lambda s: s.abs(), ascending=False)

population_change_sensitivity_audit.to_csv(OUTPUT_DIR / "population_change_sensitivity_audit.csv", index=False)
print(f"Winsorization bounds: {low_q:.4f}, {high_q:.4f}")
print(f"Correlation between original and winsorized vulnerability: {sensitivity_corr:.4f}")
display(population_change_sensitivity_audit.head(20))

Winsorization bounds: -45.2911, 93.9364
Correlation between original and winsorized vulnerability: 0.8856


,Barangay,Population_2020,Population_2024,Population_Change_2020_2024_Pct,Population_Change_2020_2024_Pct_Winsorized,Vulnerability,Vulnerability_WinsorizedPopChange,Vulnerability_Sensitivity_Diff,DPI_Scaled,DPI_Risk_Class
193,Barangay 481,1004,1943.0,93.525896,93.525896,0.424252,0.679327,0.255074,25.647750,High
627,Barangay 124,159,307.0,93.081761,93.081761,0.345678,0.599896,0.254218,17.279108,Low
643,Barangay 729,871,1775.0,103.788749,93.936411,0.353070,0.604352,0.251282,16.895220,Low
725,Barangay 8,147,301.0,104.761905,93.936411,0.340930,0.591760,0.250829,14.254624,Low
241,Barangay 271,633,1315.0,107.740916,93.936411,0.383373,0.632816,0.249443,24.254524,High
157,Barangay 296,1195,2273.0,90.209205,90.209205,0.427391,0.676068,0.248677,27.154091,High
41,Barangay 262,1736,3638.0,109.562212,93.936411,0.449207,0.697803,0.248596,35.668946,High
8,Barangay 397,4175,8835.0,111.616766,93.936411,0.525475,0.773115,0.247640,64.590671,High
441,Barangay 692,1329,2508.0,88.713318,88.713318,0.359601,0.605393,0.245791,20.255750,Medium
409,Barangay 490,471,1048.0,122.505308,93.936411,0.391169,0.633743,0.242574,20.898856,Medium


## F. Permutation Importance for Selected Model

Permutation importance estimates how much each predictor contributes to held-out F1-macro by randomly permuting one feature at a time. It complements impurity-based tree importance, which can be biased for correlated or continuous predictors.


In [38]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    n_repeats=30,
    random_state=RANDOM_STATE,
    scoring="f1_macro",
)

permutation_importance_df = pd.DataFrame({
    "Feature": ml_features,
    "Permutation_Importance_Mean": perm.importances_mean,
    "Permutation_Importance_Std": perm.importances_std,
    "Model": best_model_name,
}).sort_values("Permutation_Importance_Mean", ascending=False)

permutation_importance_df.to_csv(OUTPUT_DIR / "permutation_feature_importance.csv", index=False)
display(permutation_importance_df)

,Feature,Permutation_Importance_Mean,Permutation_Importance_Std,Model
6,Affected_Population_25yr,0.428501,0.035839,Gradient Boosting
1,Flood_PCT_25yr,0.150726,0.021186,Gradient Boosting
4,Population_2024,0.025782,0.010744,Gradient Boosting
10,Elevation_Mean,0.013421,0.007769,Gradient Boosting
7,Affected_Population_Increase_5to25,0.010081,0.010179,Gradient Boosting
9,Population_Change_2020_2024_Pct,0.008903,0.003693,Gradient Boosting
5,Affected_Population_5yr,0.006597,0.010696,Gradient Boosting
3,Flood_Severity_Increase_5to25_Sqm,0.005335,0.007879,Gradient Boosting
2,Flood_Escalation_5to25,0.001793,0.005044,Gradient Boosting
8,Population_Density_per_Hectare,0.001763,0.006403,Gradient Boosting


# Appendix | Automated PDF evidence report

This original report-generation cell is preserved. Run it **after** the analytical and diagnostic cells. It requires `reportlab` and uses the current run's variables; it should not be presented as an independent validation of observed floods. To install dependencies in your own Jupyter environment, run `%pip install reportlab` in a separate setup cell, restart the kernel, and rerun all cells.


In [39]:
# Comprehensive PDF Report Generator with Clear Tables
!pip install reportlab
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import textwrap

# ReportLab imports
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4, landscape, portrait
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_LEFT, TA_CENTER
from reportlab.lib.units import inch, cm
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    PageBreak,
    Table,
    TableStyle,
    Image,
    KeepTogether
)
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.pdfbase import pdfmetrics

# Matplotlib only for charts
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 1. Output Configuration
# ------------------------------------------------------------

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

PDF_REPORT_PATH = OUTPUT_DIR / "Manila_Barangay_Flood_Risk_Comprehensive_Report_READABLE.pdf"

GENERATED_AT = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print(f"Preparing readable PDF report: {PDF_REPORT_PATH}")


# ------------------------------------------------------------
# 2. Styles
# ------------------------------------------------------------

styles = getSampleStyleSheet()

TITLE_STYLE = ParagraphStyle(
    "CustomTitle",
    parent=styles["Title"],
    fontSize=18,
    leading=22,
    alignment=TA_CENTER,
    spaceAfter=14,
)

SUBTITLE_STYLE = ParagraphStyle(
    "CustomSubtitle",
    parent=styles["Normal"],
    fontSize=11,
    leading=15,
    alignment=TA_CENTER,
    textColor=colors.darkslategray,
    spaceAfter=18,
)

SECTION_STYLE = ParagraphStyle(
    "SectionHeading",
    parent=styles["Heading1"],
    fontSize=14,
    leading=18,
    spaceBefore=12,
    spaceAfter=8,
    textColor=colors.HexColor("#1F4E79"),
)

SUBSECTION_STYLE = ParagraphStyle(
    "SubsectionHeading",
    parent=styles["Heading2"],
    fontSize=11,
    leading=14,
    spaceBefore=8,
    spaceAfter=6,
    textColor=colors.HexColor("#2F5597"),
)

BODY_STYLE = ParagraphStyle(
    "BodyTextCustom",
    parent=styles["BodyText"],
    fontSize=9,
    leading=13,
    alignment=TA_LEFT,
    spaceAfter=8,
)

NOTE_STYLE = ParagraphStyle(
    "NoteStyle",
    parent=styles["BodyText"],
    fontSize=8,
    leading=11,
    textColor=colors.dimgray,
    leftIndent=8,
    rightIndent=8,
    spaceAfter=8,
)

TABLE_CELL_STYLE = ParagraphStyle(
    "TableCell",
    parent=styles["BodyText"],
    fontSize=6.5,
    leading=8,
    alignment=TA_CENTER,
)

TABLE_HEADER_STYLE = ParagraphStyle(
    "TableHeader",
    parent=styles["BodyText"],
    fontSize=6.5,
    leading=8,
    alignment=TA_CENTER,
    textColor=colors.white,
)

SMALL_TABLE_CELL_STYLE = ParagraphStyle(
    "SmallTableCell",
    parent=styles["BodyText"],
    fontSize=6,
    leading=7,
    alignment=TA_CENTER,
)

SMALL_TABLE_HEADER_STYLE = ParagraphStyle(
    "SmallTableHeader",
    parent=styles["BodyText"],
    fontSize=6,
    leading=7,
    alignment=TA_CENTER,
    textColor=colors.white,
)


# ------------------------------------------------------------
# 3. Helper Functions
# ------------------------------------------------------------

def safe_read_csv(path):
    path = Path(path)
    if path.exists():
        try:
            return pd.read_csv(path)
        except Exception as e:
            print(f"Warning: Could not read {path}: {e}")
            return None
    return None


def get_existing_df(var_name, fallback_path=None):
    if var_name in globals():
        obj = globals()[var_name]
        if isinstance(obj, pd.DataFrame):
            return obj.copy()

    if fallback_path is not None:
        return safe_read_csv(fallback_path)

    return None


def paragraph(text, style=BODY_STYLE):
    text = str(text)
    text = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    text = text.replace("\n", "<br/>")
    return Paragraph(text, style)


def clean_value(value, decimals=4):
    if pd.isna(value):
        return ""

    if isinstance(value, float):
        return f"{value:.{decimals}f}"

    if isinstance(value, (int, np.integer)):
        return f"{value:,}"

    value = str(value)

    # Shorten very long dictionary strings, usually best params
    if len(value) > 120:
        value = value[:117] + "..."

    return value


def prepare_table_data(df, decimals=4, max_text_len=120, small=False):
    """
    Convert a dataframe into ReportLab Paragraph table data.
    This prevents columns from being visually merged.
    """
    if df is None or df.empty:
        return None

    cell_style = SMALL_TABLE_CELL_STYLE if small else TABLE_CELL_STYLE
    header_style = SMALL_TABLE_HEADER_STYLE if small else TABLE_HEADER_STYLE

    # Header row
    data = []
    header_row = []
    for col in df.columns:
        col_text = str(col).replace("_", "_<br/>")
        header_row.append(Paragraph(col_text, header_style))
    data.append(header_row)

    # Body rows
    for _, row in df.iterrows():
        body_row = []
        for value in row:
            value_text = clean_value(value, decimals=decimals)
            if len(value_text) > max_text_len:
                value_text = value_text[:max_text_len - 3] + "..."
            value_text = value_text.replace("_", "_<br/>")
            body_row.append(Paragraph(value_text, cell_style))
        data.append(body_row)

    return data


def add_table(
    story,
    title,
    df,
    note=None,
    max_rows=25,
    preferred_cols=None,
    column_groups=None,
    repeat_header=True,
    page_break_before=True
):
    """
    Add a readable table to the PDF.

    If column_groups is provided, the dataframe is split into multiple smaller tables.
    This is the best way to avoid compressed unreadable wide tables.
    """
    if page_break_before:
        story.append(PageBreak())

    story.append(Paragraph(title, SECTION_STYLE))

    if note:
        story.append(Paragraph(note, NOTE_STYLE))

    if df is None or df.empty:
        story.append(Paragraph("No data available for this section.", BODY_STYLE))
        return

    display_df = df.copy()

    if preferred_cols is not None:
        existing_cols = [c for c in preferred_cols if c in display_df.columns]
        if existing_cols:
            display_df = display_df[existing_cols]

    if max_rows is not None and len(display_df) > max_rows:
        display_df = display_df.head(max_rows)
        story.append(Paragraph(
            f"Showing first {max_rows} rows for readability.",
            NOTE_STYLE
        ))

    # If column groups are provided, create separate tables
    if column_groups:
        for group_title, cols in column_groups:
            existing_cols = [c for c in cols if c in display_df.columns]
            if not existing_cols:
                continue

            group_df = display_df[existing_cols].copy()

            story.append(Spacer(1, 0.15 * inch))
            story.append(Paragraph(group_title, SUBSECTION_STYLE))

            table_data = prepare_table_data(group_df, small=(len(existing_cols) > 8))

            available_width = landscape(A4)[0] - 1.0 * inch
            col_width = available_width / len(existing_cols)
            col_widths = [col_width] * len(existing_cols)

            table = Table(
                table_data,
                colWidths=col_widths,
                repeatRows=1 if repeat_header else 0,
                hAlign="CENTER"
            )

            table.setStyle(get_table_style())
            story.append(table)
            story.append(Spacer(1, 0.2 * inch))

        return

    # Normal single table
    num_cols = display_df.shape[1]
    table_data = prepare_table_data(display_df, small=(num_cols > 8))

    available_width = landscape(A4)[0] - 1.0 * inch
    col_width = available_width / num_cols
    col_widths = [col_width] * num_cols

    table = Table(
        table_data,
        colWidths=col_widths,
        repeatRows=1 if repeat_header else 0,
        hAlign="CENTER"
    )

    table.setStyle(get_table_style())
    story.append(table)


def get_table_style():
    return TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#2F5597")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("ALIGN", (0, 0), (-1, -1), "CENTER"),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTNAME", (0, 1), (-1, -1), "Helvetica"),
        ("FONTSIZE", (0, 0), (-1, -1), 6),
        ("BOTTOMPADDING", (0, 0), (-1, 0), 5),
        ("TOPPADDING", (0, 0), (-1, -1), 4),
        ("BOTTOMPADDING", (0, 1), (-1, -1), 4),
        ("GRID", (0, 0), (-1, -1), 0.25, colors.grey),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#F2F2F2")]),
    ])


def add_bar_chart(story, title, labels, values, ylabel, filename, note=None):
    """
    Save a chart as PNG and add it to the PDF.
    """
    chart_path = OUTPUT_DIR / filename

    labels = [str(x) for x in labels]
    values = list(values)

    plt.figure(figsize=(10, 5.5))
    bars = plt.bar(labels, values, color="#4472C4")
    plt.title(title, fontsize=14, fontweight="bold")
    plt.ylabel(ylabel)
    plt.xticks(rotation=35, ha="right")
    plt.grid(axis="y", alpha=0.3)

    for bar, val in zip(bars, values):
        try:
            plt.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height(),
                f"{val:.4f}",
                ha="center",
                va="bottom",
                fontsize=8
            )
        except Exception:
            pass

    plt.tight_layout()
    plt.savefig(chart_path, dpi=200, bbox_inches="tight")
    plt.close()

    story.append(PageBreak())
    story.append(Paragraph(title, SECTION_STYLE))

    if note:
        story.append(Paragraph(note, NOTE_STYLE))

    story.append(Image(str(chart_path), width=9.2 * inch, height=5.0 * inch))


def add_footer(canvas, doc):
    canvas.saveState()
    canvas.setFont("Helvetica", 7)
    canvas.setFillColor(colors.grey)
    page_num = canvas.getPageNumber()
    canvas.drawCentredString(
        landscape(A4)[0] / 2,
        0.35 * inch,
        f"Generated: {GENERATED_AT} | Page {page_num}"
    )
    canvas.restoreState()


# ------------------------------------------------------------
# 4. Load Result Tables
# ------------------------------------------------------------

df_predictions = get_existing_df(
    "predictions_df",
    OUTPUT_DIR / "barangay_flood_risk_predictions.csv"
)

if df_predictions is None:
    df_predictions = get_existing_df(
        "df_predictions",
        OUTPUT_DIR / "barangay_flood_risk_predictions.csv"
    )

if df_predictions is None:
    df_predictions = get_existing_df(
        "df_ml",
        OUTPUT_DIR / "barangay_flood_risk_predictions.csv"
    )

df_model_results = get_existing_df(
    "model_results_df",
    OUTPUT_DIR / "model_comparison_results.csv"
)

if df_model_results is None:
    df_model_results = get_existing_df(
        "results_df",
        OUTPUT_DIR / "model_comparison_results.csv"
    )

df_entropy = get_existing_df(
    "entropy_weights_df",
    OUTPUT_DIR / "entropy_component_weights.csv"
)

if df_entropy is None:
    df_entropy = get_existing_df(
        "component_weights_df",
        OUTPUT_DIR / "entropy_component_weights.csv"
    )

df_gb_importance = get_existing_df(
    "gb_importance_df",
    OUTPUT_DIR / "feature_importance_gradient_boosting.csv"
)

df_rf_importance = get_existing_df(
    "rf_importance_df",
    OUTPUT_DIR / "feature_importance_random_forest.csv"
)

df_perm_importance = get_existing_df(
    "permutation_importance_df",
    OUTPUT_DIR / "permutation_feature_importance.csv"
)

df_component_summary = get_existing_df(
    "component_class_summary",
    OUTPUT_DIR / "dpi_component_class_summary.csv"
)

df_top_rank = get_existing_df(
    "top_rank_explanation",
    OUTPUT_DIR / "top_rank_explanation.csv"
)

df_high_dpi_medium_flood = get_existing_df(
    "high_dpi_medium_flood",
    OUTPUT_DIR / "high_dpi_medium_flood_audit.csv"
)

df_elevation_audit = get_existing_df(
    "elevation_outlier_audit",
    OUTPUT_DIR / "elevation_outlier_audit.csv"
)

df_population_sensitivity = get_existing_df(
    "population_change_sensitivity_audit",
    OUTPUT_DIR / "population_change_sensitivity_audit.csv"
)


# ------------------------------------------------------------
# 5. Generate Missing Summary Tables If Needed
# ------------------------------------------------------------

if df_predictions is not None and "DPI_Risk_Class" in df_predictions.columns:
    risk_class_counts = (
        df_predictions["DPI_Risk_Class"]
        .value_counts()
        .rename_axis("DPI_Risk_Class")
        .reset_index(name="Barangay_Count")
    )
else:
    risk_class_counts = None


if df_component_summary is None and df_predictions is not None and "DPI_Risk_Class" in df_predictions.columns:
    summary_cols = [
        "DPI_Scaled",
        "Flood_PCT_25yr",
        "Population_2024",
        "Affected_Population_25yr",
        "Elevation_Mean",
        "Hazard",
        "CSI",
        "Exposure",
        "Vulnerability"
    ]

    existing = [c for c in summary_cols if c in df_predictions.columns]

    agg = {
        f"Mean_{c}": (c, "mean")
        for c in existing
        if pd.api.types.is_numeric_dtype(df_predictions[c])
    }

    df_component_summary = (
        df_predictions
        .groupby("DPI_Risk_Class")
        .agg(Barangay_Count=("DPI_Risk_Class", "count"), **agg)
        .reset_index()
    )


if df_top_rank is None and df_predictions is not None:
    if "DPI_Rank" in df_predictions.columns:
        df_top_rank = df_predictions.sort_values("DPI_Rank", ascending=True).head(25)
    elif "DPI_Scaled" in df_predictions.columns:
        df_top_rank = df_predictions.sort_values("DPI_Scaled", ascending=False).head(25)


# ------------------------------------------------------------
# 6. Build the PDF Report
# ------------------------------------------------------------

doc = SimpleDocTemplate(
    str(PDF_REPORT_PATH),
    pagesize=landscape(A4),
    rightMargin=0.45 * inch,
    leftMargin=0.45 * inch,
    topMargin=0.45 * inch,
    bottomMargin=0.55 * inch
)

story = []

# ------------------------------------------------------------
# Cover Page
# ------------------------------------------------------------

story.append(Spacer(1, 1.0 * inch))
story.append(Paragraph(
    "Manila Barangay Flood Risk Priority Classification Report",
    TITLE_STYLE
))
story.append(Paragraph(
    "Disaster Prioritization Index (DPI) and Machine Learning Classification-Reproduction Results",
    SUBTITLE_STYLE
))

cover_info = """
<b>Study Area:</b> City of Manila, Philippines<br/>
<b>Unit of Analysis:</b> Barangay-level total enumeration<br/>
<b>Expected Barangay Count:</b> 897 barangays<br/>
<b>Main Outputs:</b> Disaster Prioritization Index, Low/Medium/High priority classes, machine learning model comparison, barangay ranking table, diagnostic audit tables, and feature importance results.<br/>
<b>Generated:</b> {}
""".format(GENERATED_AT)

story.append(Paragraph(cover_info, BODY_STYLE))
story.append(PageBreak())


# ------------------------------------------------------------
# Text Sections
# ------------------------------------------------------------

story.append(Paragraph("1. Project Overview", SECTION_STYLE))
story.append(Paragraph("""
This report summarizes the outputs of the Manila barangay-level flood risk priority classification pipeline. The study covers all 897 barangays of the City of Manila and integrates LiPAD 5-year and 25-year flood hazard layers, PSA 2020 and PSA 2024 population data, barangay administrative information, and DEM-derived elevation.
""", BODY_STYLE))

story.append(Paragraph("""
The 100-year flood layer is excluded from the main Disaster Prioritization Index and machine learning feature set because it failed return-period consistency checks. The Disaster Prioritization Index combines three major components: Hazard, Exposure, and Vulnerability. Component scores are normalized and combined using entropy-derived weights.
""", BODY_STYLE))

story.append(Paragraph("""
The machine learning stage is interpreted as a classification-reproduction experiment. The target variable is the DPI-derived risk class. Therefore, model performance indicates how well selected raw and engineered features reproduce the DPI-derived priority structure. It should not be interpreted as independent prediction of actual flood events, flood damage, casualties, or future flood conditions.
""", BODY_STYLE))

story.append(Spacer(1, 0.15 * inch))

story.append(Paragraph("2. Methodological Summary", SECTION_STYLE))
story.append(Paragraph("""
<b>Spatial Processing:</b> Flood polygons are intersected with barangay boundaries to compute flood area and flood percentage for 5-year and 25-year return-period scenarios. Area-related calculations are performed using EPSG:32651.
""", BODY_STYLE))
story.append(Paragraph("""
<b>Population Integration:</b> PSA 2020 and PSA 2024 barangay-level population values are integrated. Population change and population density per hectare are computed for vulnerability assessment and machine learning.
""", BODY_STYLE))
story.append(Paragraph("""
<b>Affected Population:</b> Affected population is estimated by multiplying barangay population by flood percentage. This assumes uniform population distribution within each barangay.
""", BODY_STYLE))
story.append(Paragraph("""
<b>DPI Computation:</b> Hazard, Exposure, and Vulnerability component scores are computed from normalized indicators. Entropy weighting is then applied to determine final component weights. DPI scores are scaled from 0 to 100 and classified into Low, Medium, and High priority groups using tertiles.
""", BODY_STYLE))

story.append(Spacer(1, 0.15 * inch))

story.append(Paragraph("3. Exposure-Weight Interpretation", SECTION_STYLE))
story.append(Paragraph("""
The entropy-weighted DPI may assign a larger weight to the Exposure component when affected-population values vary more strongly across barangays than the Hazard or Vulnerability components. This does not mean that hazard or vulnerability indicators are unimportant. Instead, it means that, within the Manila barangay dataset, differences in potentially affected population are a major source of variation in final priority scores.
""", BODY_STYLE))
story.append(Paragraph("""
As a result, some highly populated barangays may rank highly even if their flood coverage percentage is lower than that of smaller barangays. This is consistent with a disaster-prioritization objective, where the number of potentially affected residents is an important consideration for preparedness planning, monitoring, and resource allocation.
""", BODY_STYLE))


# ------------------------------------------------------------
# Entropy Weights
# ------------------------------------------------------------

add_table(
    story,
    "4. Entropy Component Weights",
    df_entropy,
    note="Entropy weights describe the data-driven contribution of Hazard, Exposure, and Vulnerability to the final DPI.",
    max_rows=10,
    preferred_cols=[
        "Component",
        "Entropy_Value",
        "Diversification_Value",
        "Entropy_Weight"
    ],
    page_break_before=True
)

if df_entropy is not None:
    component_col = "Component" if "Component" in df_entropy.columns else None
    weight_col = "Entropy_Weight" if "Entropy_Weight" in df_entropy.columns else None

    if component_col and weight_col:
        add_bar_chart(
            story,
            "Entropy Weights by DPI Component",
            df_entropy[component_col],
            df_entropy[weight_col],
            "Entropy Weight",
            "chart_entropy_weights.png",
            note="Higher entropy weight indicates greater component contribution based on data dispersion."
        )


# ------------------------------------------------------------
# Risk Class Distribution
# ------------------------------------------------------------

add_table(
    story,
    "5. DPI Risk Class Distribution",
    risk_class_counts,
    note="Tertile classification produces equal Low, Medium, and High priority groups.",
    max_rows=10,
    preferred_cols=[
        "DPI_Risk_Class",
        "Barangay_Count"
    ],
    page_break_before=True
)

if risk_class_counts is not None:
    add_bar_chart(
        story,
        "DPI Risk Class Counts",
        risk_class_counts["DPI_Risk_Class"],
        risk_class_counts["Barangay_Count"],
        "Number of Barangays",
        "chart_risk_class_counts.png",
        note="The classes are relative priority tertiles within Manila."
    )


# ------------------------------------------------------------
# Component Summary
# ------------------------------------------------------------

component_summary_groups = [
    (
        "DPI and Count",
        [
            "DPI_Risk_Class",
            "Barangay_Count",
            "Mean_DPI",
            "Min_DPI",
            "Max_DPI"
        ]
    ),
    (
        "Component and Hazard Summary",
        [
            "DPI_Risk_Class",
            "Mean_Hazard",
            "Mean_CSI",
            "Mean_Exposure",
            "Mean_Vulnerability",
            "Mean_Flood_25yr"
        ]
    ),
    (
        "Population and Elevation Summary",
        [
            "DPI_Risk_Class",
            "Mean_Population_2024",
            "Mean_Affected_Population_25yr",
            "Mean_Elevation"
        ]
    )
]

add_table(
    story,
    "6. DPI Component and Class Summary",
    df_component_summary,
    note="Wide tables are split into smaller sub-tables to improve readability.",
    max_rows=10,
    column_groups=component_summary_groups,
    page_break_before=True
)


# ------------------------------------------------------------
# Top-Ranked Barangays
# ------------------------------------------------------------

top_rank_groups = [
    (
        "Identification and Ranking",
        [
            "DPI_Rank",
            "Barangay",
            "Barangay_No",
            "DPI_Risk_Class",
            "DPI_Scaled"
        ]
    ),
    (
        "Population and Flood Exposure",
        [
            "Barangay",
            "Population_2024",
            "Flood_PCT_25yr",
            "Affected_Population_25yr"
        ]
    ),
    (
        "Component Scores and Elevation",
        [
            "Barangay",
            "Elevation_Mean",
            "Hazard",
            "CSI",
            "Exposure",
            "Vulnerability"
        ]
    )
]

add_table(
    story,
    "7. Top-Ranked Barangays by DPI",
    df_top_rank,
    note="High-ranking barangays may be driven by flood coverage, high exposed population, low elevation, density, or combined factors.",
    max_rows=25,
    column_groups=top_rank_groups,
    page_break_before=True
)


# ------------------------------------------------------------
# High-DPI medium-Flood Audit
# ------------------------------------------------------------

high_dpi_groups = [
    (
        "Identification and DPI",
        [
            "DPI_Rank",
            "Barangay",
            "DPI_Risk_Class",
            "DPI_Scaled"
        ]
    ),
    (
        "Flood and Exposure",
        [
            "Barangay",
            "Population_2024",
            "Flood_PCT_25yr",
            "Affected_Population_25yr",
            "Exposure"
        ]
    ),
    (
        "Vulnerability Context",
        [
            "Barangay",
            "Elevation_Mean",
            "Hazard",
            "CSI",
            "Vulnerability"
        ]
    )
]

add_table(
    story,
    "8. High-DPI but medium-Flood Audit",
    df_high_dpi_medium_flood,
    note="These barangays are classified as High priority despite below-median 25-year flood percentage. This often reflects high exposed population or vulnerability factors.",
    max_rows=25,
    column_groups=high_dpi_groups,
    page_break_before=True
)


# ------------------------------------------------------------
# Elevation Outlier Audit
# ------------------------------------------------------------

elevation_groups = [
    (
        "Elevation Statistics",
        [
            "Barangay",
            "Barangay_No",
            "Elevation_Mean",
            "Elevation_Median",
            "Elevation_Min",
            "Elevation_Max",
            "Elevation_Count"
        ]
    ),
    (
        "Flood and Population Context",
        [
            "Barangay",
            "Flood_PCT_25yr",
            "Population_2024",
            "Affected_Population_25yr"
        ]
    ),
    (
        "DPI Context",
        [
            "Barangay",
            "Vulnerability",
            "Exposure",
            "DPI_Scaled",
            "DPI_Risk_Class"
        ]
    )
]

add_table(
    story,
    "9. Elevation Outlier Audit",
    df_elevation_audit,
    note="Elevation outliers should be retained unless confirmed as data errors, but they should be interpreted carefully in relation to exposure and vulnerability.",
    max_rows=20,
    column_groups=elevation_groups,
    page_break_before=True
)


# ------------------------------------------------------------
# Population Change Sensitivity Audit
# ------------------------------------------------------------

population_groups = [
    (
        "Population Change",
        [
            "Barangay",
            "Population_2020",
            "Population_2024",
            "Population_Change_2020_2024_Pct",
            "Population_Change_2020_2024_Pct_Winsorized"
        ]
    ),
    (
        "Vulnerability Sensitivity",
        [
            "Barangay",
            "Vulnerability",
            "Vulnerability_WinsorizedPopChange",
            "Vulnerability_Sensitivity_Diff"
        ]
    ),
    (
        "DPI Context",
        [
            "Barangay",
            "DPI_Scaled",
            "DPI_Risk_Class"
        ]
    )
]

add_table(
    story,
    "10. Population-Change Sensitivity Audit",
    df_population_sensitivity,
    note="This diagnostic checks whether extreme population-change percentages materially affect the Vulnerability component.",
    max_rows=25,
    column_groups=population_groups,
    page_break_before=True
)


# ------------------------------------------------------------
# Model Comparison
# ------------------------------------------------------------

model_groups = [
    (
        "Cross-Validation Performance",
        [
            "Model",
            "CV_F1_Macro_Mean",
            "CV_F1_Macro_Std",
            "Best_Params"
        ]
    ),
    (
        "Held-Out Test Performance",
        [
            "Model",
            "Accuracy",
            "Balanced_Accuracy",
            "F1_Macro",
            "F1_Weighted",
            "ROC_AUC_OVR_Macro"
        ]
    ),
    (
        "High-Class Performance",
        [
            "Model",
            "Precision_High",
            "Recall_High",
            "F1_High",
            "Support_High"
        ]
    ),
    (
        "Low-Class Performance",
        [
            "Model",
            "Precision_Low",
            "Recall_Low",
            "F1_Low",
            "Support_Low"
        ]
    ),
    (
        "Medium-Class Performance",
        [
            "Model",
            "Precision_Medium",
            "Recall_Medium",
            "F1_Medium",
            "Support_Medium"
        ]
    )
]

add_table(
    story,
    "11. Machine Learning Model Comparison",
    df_model_results,
    note="Model performance reflects classification-reproduction of DPI-derived risk classes, not independent prediction of real flood outcomes.",
    max_rows=10,
    column_groups=model_groups,
    page_break_before=True
)

if df_model_results is not None and "Model" in df_model_results.columns:
    metric_col = None
    for candidate in ["CV_F1_Macro_Mean", "F1_Macro", "Accuracy", "Balanced_Accuracy"]:
        if candidate in df_model_results.columns:
            metric_col = candidate
            break

    if metric_col:
        add_bar_chart(
            story,
            f"Model Comparison by {metric_col}",
            df_model_results["Model"],
            df_model_results[metric_col],
            metric_col,
            "chart_model_comparison.png",
            note="The selected model should be based on the predefined model-selection metric, typically cross-validation F1-macro."
        )


# ------------------------------------------------------------
# Feature Importance Tables
# ------------------------------------------------------------

add_table(
    story,
    "12. Gradient Boosting Feature Importance",
    df_gb_importance,
    note="Impurity-based feature importance explains how the Gradient Boosting model reproduced DPI-derived classes.",
    max_rows=20,
    preferred_cols=[
        "Feature",
        "Importance",
        "Model"
    ],
    page_break_before=True
)

if df_gb_importance is not None and {"Feature", "Importance"}.issubset(df_gb_importance.columns):
    gb_plot_df = df_gb_importance.sort_values("Importance", ascending=False).head(10)
    add_bar_chart(
        story,
        "Top Gradient Boosting Feature Importances",
        gb_plot_df["Feature"],
        gb_plot_df["Importance"],
        "Importance",
        "chart_gb_importance.png",
        note="Higher values indicate stronger contribution to the model's classification-reproduction behavior."
    )


add_table(
    story,
    "13. Random Forest Feature Importance",
    df_rf_importance,
    note="Impurity-based feature importance from the Random Forest model.",
    max_rows=20,
    preferred_cols=[
        "Feature",
        "Importance",
        "Model"
    ],
    page_break_before=True
)

if df_rf_importance is not None and {"Feature", "Importance"}.issubset(df_rf_importance.columns):
    rf_plot_df = df_rf_importance.sort_values("Importance", ascending=False).head(10)
    add_bar_chart(
        story,
        "Top Random Forest Feature Importances",
        rf_plot_df["Feature"],
        rf_plot_df["Importance"],
        "Importance",
        "chart_rf_importance.png",
        note="Feature importance should be interpreted cautiously when predictors are correlated."
    )


add_table(
    story,
    "14. Permutation Importance",
    df_perm_importance,
    note="Permutation importance estimates how much each feature contributes to held-out F1-macro when feature values are randomly shuffled.",
    max_rows=20,
    preferred_cols=[
        "Feature",
        "Permutation_Importance_Mean",
        "Permutation_Importance_Std",
        "Model"
    ],
    page_break_before=True
)

if df_perm_importance is not None:
    feature_col = "Feature" if "Feature" in df_perm_importance.columns else None
    perm_col = "Permutation_Importance_Mean" if "Permutation_Importance_Mean" in df_perm_importance.columns else None

    if feature_col and perm_col:
        perm_plot_df = df_perm_importance.sort_values(perm_col, ascending=False).head(10)
        add_bar_chart(
            story,
            "Top Permutation Importances",
            perm_plot_df[feature_col],
            perm_plot_df[perm_col],
            "Permutation Importance",
            "chart_perm_importance.png",
            note="Permutation importance is model-agnostic and helps validate impurity-based feature-importance patterns."
        )


# ------------------------------------------------------------
# Limitations
# ------------------------------------------------------------

story.append(PageBreak())
story.append(Paragraph("15. Limitations and Usage Notes", SECTION_STYLE))

limitations = [
    "The study relies on secondary datasets and does not include field validation, household surveys, or complete barangay-level historical damage records.",
    "LiPAD flood layers represent modeled hazard scenarios and may not fully reflect real-time flooding, drainage constraints, rainfall intensity, tidal influence, pumping conditions, land-use changes, or recent infrastructure updates.",
    "The 100-year flood layer is excluded from the main analysis due to return-period consistency anomalies. Any retained 100-year fields in intermediate files are for traceability only.",
    "Affected population is estimated using barangay-level flood percentage and total population. This assumes a uniform distribution of residents within each barangay, which may not fully reflect actual settlement patterns.",
    "Elevation is summarized at the barangay level and does not capture micro-scale road depressions, building elevation, local drainage obstructions, land subsidence, or flood depth.",
    "Low, Medium, and High classes are relative tertiles within Manila. They are not official flood-warning categories or externally calibrated hazard levels.",
    "Machine learning performance measures classification-reproduction of DPI-derived classes. The models do not independently predict actual flood occurrence, flood depth, damage, casualties, or future flood conditions.",
    "The prototype and outputs are intended for research and planning support only. They should not replace official government hazard advisories, evacuation orders, or operational disaster response systems."
]

for i, item in enumerate(limitations, start=1):
    story.append(Paragraph(f"<b>{i}.</b> {item}", BODY_STYLE))


# ------------------------------------------------------------
# 7. Build PDF
# ------------------------------------------------------------

doc.build(story, onFirstPage=add_footer, onLaterPages=add_footer)

print("=" * 80)
print("Readable PDF report successfully generated.")
print(f"Saved to: {PDF_REPORT_PATH.resolve()}")
print("=" * 80)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 22.7 MB/s eta 0:00:00
Preparing readable PDF report: outputs/Manila_Barangay_Flood_Risk_Comprehensive_Report_READABLE.pdf
Readable PDF report successfully generated.
Saved to: /content/outputs/Manila_Barangay_Flood_Risk_Comprehensive_Report_READABLE.pdf
